# ConvDecoder — Multicoil Knee MRI Reconstruction

This notebook fits ConvDecoder to undersampled multicoil fastMRI knee data. It runs a controlled comparison of initialization strategies, input codes (uniform noise vs. VAE-encoded), architectures (plain vs. SE-attention), and loss functions (MSE vs. Huber+TV), then batch-evaluates the K-averaged ConvDecoder-A ensemble and plain Run A/B/C fits over many images for paper-style mean ± std results.

## How to run this notebook

**Prerequisites:** a Google Drive containing fastMRI knee multicoil *validation* `.h5` files under `/content/drive/MyDrive/Projects/knee_multicoil_val/multicoil_val` (Section 6 expects this exact path). A GPU runtime — T4/L4/A100/G4 all work; the per-section cost estimates below assume these.

**Recommended order for a full reproduction:**
1. Run Sections 1–9 top to bottom once per `Z_SOURCE` / `ARCHITECTURE` / loss config combination you want single-image results for (see the toggle table below). `RUN_MODE="load_existing"` makes re-running safe and cheap once a configuration is cached.
2. Use Sections 10–11 for pairwise Δ comparisons between two already-run configurations.
3. Use Section 12 (set `K_AVERAGING=True`) for the k-averaged ConvDecoder-A ensemble, and Section 12.5 (`RUN_BATCH_EVAL=True`) to batch it over many images.
4. Use Section 13 (`RUN_ABC_BATCH_EVAL=True`) for the single-fit Run A/B/C batch evaluation. **This needs Section 12.5.1 to have executed at least once first** (even with `K_AVERAGING=False`), since it reuses that section's selected image list so every method is compared on identical images — see Section 13's own note if you hit an assertion error here.
5. (Optional) Use Section 15 to snapshot a milestone: copies the small results CSVs and this notebook (outputs stripped) into the GitHub repo and pushes. Checkpoints and the full Drive results tree are never pushed — only code + small summaries.

**All toggles live in Section 2.** Everything downstream (checkpoint paths, result filenames) is auto-tagged by their values, so re-running with a different toggle combination never collides with or silently overwrites a previous run's saved output.

| Toggle | What it controls | Typical values |
|---|---|---|
| `Z_SOURCE` | fixed input code `z`: uniform noise vs. pretrained MRI-VAE encoding | `"uniform"`, `"mri_vae"` |
| `ARCHITECTURE` | baseline ConvDecoder vs. SE-attention variant | `"convdecoder"`, `"convdecoder_se"` |
| `RUN_MODE` | reload a cached checkpoint vs. always refit from scratch | `"load_existing"` (default, safe), `"new_fit"` |
| `LOSS_TYPE` / `HUBER_DELTA` / `TV_WEIGHT` | data-fidelity loss + optional TV regularizer (Section 5.1) | `"mse"` (original), or `"huber"` + a delta/weight |
| `K_AVERAGING` / `K_VALUE` | Section 12: fit a `k`-member ConvDecoder-A ensemble | `True`/`False`, `k=10` |
| `RUN_BATCH_EVAL` / `KAVG_BATCH_SIZE` | Section 12.5: batch-evaluate ConvDecoder-A over `N` images | `True`/`False`, `N=20` |
| `KAVG_BATCH_MANUAL_EXCLUDE` | filenames to permanently drop from the batch (e.g. bad reconstructions) | a `set()` of `.h5` filenames |
| `RUN_ABC_BATCH_EVAL` / `ABC_BATCH_RUNS` | Section 13: batch-evaluate single Run A/B/C over the same `N` images | `True`/`False`, subset of `{"A","B","C"}` |

**Where results land:** every run's metrics are appended to a CSV under `RESULTS_DIR` (`.../Projects/results`), and every fitted network is checkpointed under a `checkpoints_multicoil_*` directory in the same Drive project folder — both auto-tagged by the current toggle combination.

**Section map:**

1. Setup — clone repo, mount Drive, install dependencies
2. Experiment Config — every toggle described above
3. Network architecture (`build_network`)
4. Input code `z` source (`get_z`)
5. Helper functions (forward model, scale factor, metrics); 5.1 custom loss / TV regularizer
6. Load data samples (5 hand-picked demo images)
7. Reference image full fit → becomes the guided-init checkpoint for Section 8
8. Run A/B/C on a second image: random vs. guided init, full (10k-iter) vs. accelerated (1k-iter)
9. Results table for this run
10. Cross-`Z_SOURCE`/`ARCHITECTURE` Δ comparison (after running Section 1–9 under two configs)
11. Cross-loss-config Δ comparison (MSE vs. Huber/+TV, same architecture)
12. ConvDecoder-A ensemble averaging — single image (12.1–12.4), then batch over `N` images (12.5)
13. Run A/B/C batch evaluation over the same `N` images, for a fair, paired comparison against Section 12.5's ConvDecoder-A batch result
14. Delete selected fitted images safely (Drive cleanup utility)
15. Save & push results/notebook to GitHub (manual, optional — see Section 1 for the required `GITHUB_TOKEN` secret)

---

## Original walkthrough (Sections 1–9 core comparison)

**This version replaces the two separate (VAE / non-VAE) notebooks with a single toggleable one.** Set `Z_SOURCE` and `ARCHITECTURE` in the Config cell below, then run the whole notebook top to bottom. Checkpoints and results are automatically tagged by the toggle values, so different configurations never collide or silently overwrite each other.

1. **Reference fit** — a full, random-initialization fit on one reference image (`file1000033.h5`), establishing the **guided-init checkpoint** for the current toggle configuration.
2. **Three-way comparison** on a second image (`file1000041.h5`), using the *same* mask/measurement throughout:
   - Run A: random init, 10,000 iterations (full-convergence baseline)
   - Run B: guided init, 10,000 iterations (does guided-init also raise the ceiling?)
   - Run C: guided init, 1,000 iterations (does guided-init match A's quality in 1/10th the time?)
3. **Results table** — saved per-run to Drive, tagged by architecture + z-source.
4. **Comparison section** — once you've run this notebook with more than one `Z_SOURCE` (or `ARCHITECTURE`) for the same images, this section auto-loads every saved run and computes the Δ between them. No hardcoded baseline numbers.

All reconstructions are compared against ground truth and the zero-filled baseline using a standard set of MRI reconstruction quality metrics.


## 1. Setup

Clone the repo, mount Drive, install dependencies, import everything.

In [ ]:
# GPU check
import torch, os, subprocess
print("CUDA available:", torch.cuda.is_available())

# Clone (or pull, if this runtime already has it -- e.g. after a Colab disconnect/reconnect
# rather than a full factory reset) the repo. GIT_TERMINAL_PROMPT=0 makes a missing/expired
# credential fail fast with an error instead of hanging silently waiting for a prompt that
# Colab can't answer. Uses subprocess (not `!git`) so the clone-or-pull branch is plain,
# testable Python rather than shell magic inside an if/else.
os.environ["GIT_TERMINAL_PROMPT"] = "0"
REPO_DIR = "/content/ConvDeconv-MRI-project"
REPO_SLUG = "itaipasternak-cloud/ConvDeconv-MRI-project"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"{REPO_DIR} already exists -- pulling latest instead of re-cloning.")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", f"https://github.com/{REPO_SLUG}.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Extra dependency for VIF metric
!pip install sewar pytorch_msssim sigpy -q

In [ ]:
# --- GitHub push authentication (optional; only needed if you plan to run Section 15) ---
# Reads a Personal Access Token from Colab's Secrets panel (key icon, left sidebar) named
# GITHUB_TOKEN, with push access to the repo above. The token is deliberately NOT written into
# git's remote config (which would leak it into any future `git remote -v` output) -- it's only
# ever held in memory here and used fresh at push time in Section 15.
def _github_push_url():
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if not token:
        raise RuntimeError(
            "No GITHUB_TOKEN secret found. Add one in the Colab Secrets panel (key icon, left "
            "sidebar) named GITHUB_TOKEN with 'repo' push access, enable notebook access for it, "
            "then re-run this cell."
        )
    return f"https://{token}@github.com/{REPO_SLUG}.git"

import subprocess
subprocess.run(["git", "-C", REPO_DIR, "config", "user.email", "itaipasternak@gmail.com"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "config", "user.name", "Itai Pasternak"], check=True)

try:
    _github_push_url()
    print("GITHUB_TOKEN found -- Section 15 (save & push) is ready to use.")
except RuntimeError as e:
    print(f"{e}\n(Everything above Section 15 works fine without this -- it's only needed to push.)")


In [ ]:
import sys
sys.path.append('/content/ConvDeconv-MRI-project')
from demo_helper.helpers import *
from demo_helper.fit_multicoil import fit

import os
import glob
import random
import h5py
import time
import copy
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
import matplotlib.pyplot as plt
from torch.autograd import Variable
import warnings
warnings.filterwarnings('ignore')

# Metrics
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from scipy.ndimage import gaussian_laplace
from sewar.full_ref import vifp
import pytorch_msssim
import sigpy as sp
import sigpy.mri as mri

torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = True
dtype = torch.cuda.FloatTensor
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
print("num GPUs:", torch.cuda.device_count())

## 2. Experiment Config — TOGGLE HERE

`Z_SOURCE`, `ARCHITECTURE`, `RUN_MODE`, and `LOSS_TYPE`/`TV_WEIGHT` are the knobs for this notebook. Everything downstream (checkpoint paths, result filenames, plot titles) is automatically tagged from `Z_SOURCE`/`ARCHITECTURE`/loss config, so switching any value never collides with a previous run's saved output.

`RUN_MODE` controls whether the reference fit and Run A/B/C actually re-fit from scratch, or just reload and reconstruct from a previously saved checkpoint for the current configuration:
- `"load_existing"` — if a checkpoint already exists for this exact combination, load it and reconstruct (seconds, no GPU fitting). If no checkpoint exists yet, automatically falls back to a new fit, so this is safe to leave on by default.
- `"new_fit"` — always fits from scratch, ignoring any saved checkpoint (use this if you want to intentionally re-run/overwrite a configuration).

`LOSS_TYPE`/`HUBER_DELTA`/`TV_WEIGHT` (see Section 5.1) control the data-fidelity loss and optional Total Variation regularizer used during fitting. The default (`LOSS_TYPE="mse"`, `TV_WEIGHT=0`) reproduces the original ConvDecoder loss exactly and keeps the original `RUN_TAG` naming so existing checkpoints/results are still found; changing either value adds a suffix to `RUN_TAG` so it never collides with the original MSE runs.

To add a new architecture later: add its name to the assert list below, implement it in Section 3 (`build_network`), and everything else (checkpointing, results, comparison) works without further changes.

In [ ]:
# ============================================================
# EXPERIMENT CONFIG
# ============================================================
Z_SOURCE = "uniform"              # "uniform"  or  "mri_vae"      — source of the fixed input code z
ARCHITECTURE = "convdecoder"  # "convdecoder"  or  "convdecoder_se"
RUN_MODE = "load_existing"        # "load_existing"  or  "new_fit"
                                    # "load_existing": reload + reconstruct from a saved checkpoint for
                                    #   this exact configuration if one exists (fast, no fitting). Falls
                                    #   back to a new fit automatically if none is found.
                                    # "new_fit": always fits from scratch, ignoring any saved checkpoint.

# --- Loss function / regularization (see Section 5.1 for the fit() override that uses these) ---
LOSS_TYPE = "huber"    # "mse"  or  "huber"   — data-fidelity term used during fitting
HUBER_DELTA = 0.02327     # only used when LOSS_TYPE == "huber": transition point between L2/L1 behavior
TV_WEIGHT = 5.865e-05  # 0 disables Total Variation image regularization; try small values (1e-5 to 1e-3)
                        # and check that the reconstruction doesn't get visibly over-smoothed

SEED = 0                       # controls network init + mask generation reproducibility

K_AVERAGING = False    # set True to run Section 12 (ConvDecoder-A ensemble averaging): fits
                        # K independent accelerated (1,000-iter) reconstructions of the eval
                        # target, each guided-init from a DIFFERENT reference image, then
                        # averages them pixel-wise. Only applies to the accelerated regime.
K_VALUE = 10             # number of ensemble members (only used if K_AVERAGING). Matches the
                        # paper's k=5-10 averaging experiment; k=5 chosen here to bound cost.

RUN_BATCH_EVAL = False  # set True to also run Section 12.5: batch-evaluate ConvDecoder-A over
                        # KAVG_BATCH_SIZE additional images (paper-style mean +/- std), reusing the
                        # SAME K_VALUE reference checkpoints fit/loaded in Section 12.1 (no refitting).
KAVG_BATCH_SIZE = 20    # number of (new, non-reference) evaluation images for Section 12.5.
KAVG_BUILD_REF_CHECKPOINTS = False
                        # set False to make Sections 12.1-12.4 and 12.5.2-12.5.3 (everything that
                        # actually FITS a K-averaged reconstruction) no-op and print a skip message,
                        # while kavg01code (reference NAMES) and kavgbatch01code (batch target
                        # NAMES) still run normally. Use this when you only want Section 13's plain
                        # Run A/B/C batch eval for a config and don't want K_AVERAGING=True to
                        # trigger a "Run All" into K reference-checkpoint fitting you don't need.
KAVG_BATCH_MANUAL_EXCLUDE = {"file1000625.h5", "file1001168.h5", "file1000328.h5", "file1001148.h5", "file1001163.h5", "file1001331.h5"}
                        # filenames to permanently drop from the batch (e.g. qualitatively
                        # problematic reconstructions you've eyeballed) -- Section 12.5.1 excludes
                        # them from selection AND prunes any existing rows for them out of the
                        # results CSV, so re-running auto-backfills replacements to reach
                        # KAVG_BATCH_SIZE again. Leave as set() to disable.

RUN_ABC_BATCH_EVAL = False   # set True to run Section 13: evaluate Run A/B/C -- SINGLE fits, NOT
                        # k-averaged ensembles -- on the SAME batch_target_filenames selected in
                        # Section 12.5.1, reporting mean +/- std per run so every method (including
                        # K-averaged ConvDecoder-A) is compared on identical images. Requires
                        # Section 12.5.1 to have been run at least once (K_AVERAGING=True,
                        # RUN_BATCH_EVAL=True) so batch_target_filenames exists -- K_AVERAGING can
                        # be False when this actually runs.
ABC_BATCH_RUNS = {"A", "B", "C"}   # which of Run A/B/C to batch-evaluate; drop a letter to skip
                        # the most expensive ones (A and B are full 10,000-iteration fits per
                        # image; C is the cheap 1,000-iteration accelerated fit).

assert Z_SOURCE in ("uniform", "mri_vae"), f"Unknown Z_SOURCE: {Z_SOURCE!r}"
assert ARCHITECTURE in ("convdecoder", "convdecoder_se"), (
    f"Unknown ARCHITECTURE: {ARCHITECTURE!r}. "
    f"Add it to this assert list once you've implemented it in build_network() (Section 3)."
)
assert RUN_MODE in ("load_existing", "new_fit"), f"Unknown RUN_MODE: {RUN_MODE!r}"
assert LOSS_TYPE in ("mse", "huber"), f"Unknown LOSS_TYPE: {LOSS_TYPE!r}"
assert TV_WEIGHT >= 0, f"TV_WEIGHT must be >= 0, got {TV_WEIGHT!r}"
assert K_VALUE >= 1, f"K_VALUE must be >= 1, got {K_VALUE!r}"
assert KAVG_BATCH_SIZE >= 1, f"KAVG_BATCH_SIZE must be >= 1, got {KAVG_BATCH_SIZE!r}"
assert ABC_BATCH_RUNS <= {"A", "B", "C"}, (
    f"ABC_BATCH_RUNS must be a subset of {{'A','B','C'}}, got {ABC_BATCH_RUNS!r}"
)

# LOSS_TAG stays empty for the original mse/no-regularization config, so checkpoints/results saved
# before this toggle existed are still found under the same RUN_TAG. Any change to LOSS_TYPE,
# HUBER_DELTA, or TV_WEIGHT gets its own suffix so it never collides with (or silently overwrites)
# a run with a different delta/weight — HUBER_DELTA in particular used to be left OUT of this tag,
# which meant two runs with the same LOSS_TYPE/TV_WEIGHT but different HUBER_DELTA would silently
# share (and overwrite) the same checkpoint directory. See the note below load_checkpoint_guarded().
if LOSS_TYPE == "mse" and TV_WEIGHT == 0:
    LOSS_TAG = ""
else:
    delta_tag = f"_delta{HUBER_DELTA:g}" if LOSS_TYPE == "huber" else ""
    LOSS_TAG = f"_{LOSS_TYPE}{delta_tag}" + (f"_tv{TV_WEIGHT:g}" if TV_WEIGHT > 0 else "")

RUN_TAG = f"{ARCHITECTURE}_{Z_SOURCE}{LOSS_TAG}"
Z_LABEL = {"uniform": "uniform z", "mri_vae": "Microsoft MRI-VAE z"}[Z_SOURCE]
LOSS_LABEL = "MSE" if LOSS_TYPE == "mse" else f"Huber(delta={HUBER_DELTA:g})"
if TV_WEIGHT > 0:
    LOSS_LABEL += f"+TV({TV_WEIGHT:g})"

# Fixed network hyperparameters (shape-independent; per-image shape info is computed
# separately in Sections 5/6 since it depends on each file's coil count).
num_layers = 7
num_channels = 256
in_size = [8, 4]

CKPT_ROOT = "/content/drive/MyDrive/Projects"
REF_CKPT_DIR  = f"{CKPT_ROOT}/checkpoints_multicoil_{RUN_TAG}"
REF_CKPT_PATH = f"{REF_CKPT_DIR}/guided_init_multicoil_{RUN_TAG}.pt"
RUNA_CKPT_DIR = f"{CKPT_ROOT}/checkpoints_multicoil_runA_{RUN_TAG}"
RUNB_CKPT_DIR = f"{CKPT_ROOT}/checkpoints_multicoil_runB_{RUN_TAG}"
RUNC_CKPT_DIR = f"{CKPT_ROOT}/checkpoints_multicoil_runC_{RUN_TAG}"
RESULTS_DIR   = f"{CKPT_ROOT}/results"

RUNA_CKPT_PATH = f"{RUNA_CKPT_DIR}/final_runA_{RUN_TAG}.pt"
RUNB_CKPT_PATH = f"{RUNB_CKPT_DIR}/final_runB_{RUN_TAG}.pt"
RUNC_CKPT_PATH = f"{RUNC_CKPT_DIR}/final_runC_{RUN_TAG}.pt"

for d in (REF_CKPT_DIR, RUNA_CKPT_DIR, RUNB_CKPT_DIR, RUNC_CKPT_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

def set_seed(seed=SEED):
    '''Reset every RNG source. Call this immediately before anything whose randomness
    you want reproducible/matched across Z_SOURCE or ARCHITECTURE toggles (network init,
    mask generation).'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def load_checkpoint_guarded(path, expect_z_source=None, expect_architecture=None, **expected_identity):
    '''Load a checkpoint and verify it matches the current Z_SOURCE/ARCHITECTURE toggle
    (or explicit overrides), so a load_existing run never silently reconstructs from a
    mismatched configuration.'''
    ckpt = torch.load(path, weights_only=False)
    exp_z = Z_SOURCE if expect_z_source is None else expect_z_source
    exp_arch = ARCHITECTURE if expect_architecture is None else expect_architecture
    assert ckpt.get('z_source', exp_z) == exp_z, (
        f"{path} was built with z_source={ckpt.get('z_source')!r}, but expected {exp_z!r}."
    )
    assert ckpt.get('architecture', exp_arch) == exp_arch, (
        f"{path} was built with architecture={ckpt.get('architecture')!r}, but expected {exp_arch!r}."
    )
    if exp_z == "mri_vae":
        expected_model_id = "microsoft/mri-autoencoder-v0.1"
        assert ckpt.get('z_model_id') == expected_model_id, (
            f"{path} does not identify the required pretrained MRI encoder "
            f"({expected_model_id}). Delete/refit this stale checkpoint."
        )
    assert ckpt.get('loss_type', LOSS_TYPE) == LOSS_TYPE, (
        f"{path} was built with loss_type={ckpt.get('loss_type')!r}, but the current toggle is "
        f"LOSS_TYPE={LOSS_TYPE!r}."
    )
    if LOSS_TYPE == "huber":
        ckpt_delta = ckpt.get('huber_delta', HUBER_DELTA)
        assert ckpt_delta == HUBER_DELTA, (
            f"{path} was built with huber_delta={ckpt_delta!r}, but the current toggle is "
            f"HUBER_DELTA={HUBER_DELTA!r}. (This checkpoint predates HUBER_DELTA being part of "
            f"RUN_TAG, or was copied from a differently-tagged directory.)"
        )
    ckpt_tv = ckpt.get('tv_weight', TV_WEIGHT)
    assert ckpt_tv == TV_WEIGHT, (
        f"{path} was built with tv_weight={ckpt_tv!r}, but the current toggle is TV_WEIGHT={TV_WEIGHT!r}."
    )
    # New checkpoints carry stable identity metadata. Older checkpoints in the established
    # checkpoints_multicoil_<RUN_TAG> directories remain loadable: a field is checked when
    # present, while the caller-selected legacy filename supplies the missing identity.
    for key, expected in expected_identity.items():
        if expected is not None and key in ckpt:
            assert ckpt[key] == expected, (
                f"{path} has {key}={ckpt[key]!r}, but this reconstruction expects {expected!r}."
            )

    non_finite = [
        name for name, tensor in ckpt.get('model_state_dict', {}).items()
        if torch.is_tensor(tensor) and not torch.isfinite(tensor).all()
    ]
    assert not non_finite, (
        f"{path} contains NaN/Inf weights in {len(non_finite)} tensor(s) (e.g. {non_finite[0]!r}) "
        f"-- this checkpoint came from a fit that diverged (see get_scale_factor()'s finite check) "
        f"and must not be silently reused. Delete this file and re-run to refit it cleanly."
    )
    return ckpt

def stable_stem(filename):
    """Filesystem-safe, order-independent identity for an MRI file."""
    return os.path.splitext(os.path.basename(filename))[0]

def first_existing_path(*paths):
    """Return the first existing candidate, enabling transparent legacy checkpoint reuse."""
    return next((p for p in paths if p and os.path.exists(p)), None)

def checkpoint_metadata(**extra):
    """Metadata shared by every newly saved checkpoint."""
    meta = {
        'run_tag': RUN_TAG, 'z_source': Z_SOURCE,
        'z_model_id': (MRI_VAE_MODEL_ID if Z_SOURCE == 'mri_vae' else None),
        'architecture': ARCHITECTURE, 'seed': SEED,
        'loss_type': LOSS_TYPE,
        'huber_delta': HUBER_DELTA if LOSS_TYPE == 'huber' else None,
        'tv_weight': TV_WEIGHT,
    }
    meta.update(extra)
    return meta

def assert_run_tag_current():
    '''Guards against the Config cell being edited (ARCHITECTURE/Z_SOURCE/LOSS_TYPE/TV_WEIGHT
    changed) without being re-run. RUN_TAG is only recomputed when this cell actually executes, so
    if you change one of those values elsewhere (e.g. a hyperparameter search that sets HUBER_DELTA/
    TV_WEIGHT directly) and forget to re-run this cell, RUN_TAG silently stays stale and any fit/save
    that follows will write to the OLD path, overwriting whatever was there. Call this at the top of
    any cell that fits or saves a checkpoint/results file, right before it does so.'''
    expected_delta_tag = f"_delta{HUBER_DELTA:g}" if LOSS_TYPE == "huber" else ""
    expected_loss_tag = "" if (LOSS_TYPE == "mse" and TV_WEIGHT == 0) else (
        f"_{LOSS_TYPE}{expected_delta_tag}" + (f"_tv{TV_WEIGHT:g}" if TV_WEIGHT > 0 else "")
    )
    expected_run_tag = f"{ARCHITECTURE}_{Z_SOURCE}{expected_loss_tag}"
    assert expected_run_tag == RUN_TAG, (
        f"RUN_TAG ({RUN_TAG!r}) does not match what the CURRENT config would produce "
        f"({expected_run_tag!r}). A config value (ARCHITECTURE/Z_SOURCE/LOSS_TYPE/HUBER_DELTA/"
        f"TV_WEIGHT) changed without re-running the Config cell (Section 2). Re-run it now, "
        f"then re-run this cell, before fitting or saving anything."
    )

print(f"Run tag: {RUN_TAG}  |  z source: {Z_LABEL}  |  architecture: {ARCHITECTURE}  |  "
      f"run mode: {RUN_MODE}  |  loss: {LOSS_TYPE}"
      f"{f' (delta={HUBER_DELTA})' if LOSS_TYPE == 'huber' else ''}  |  TV weight: {TV_WEIGHT}  |  seed: {SEED}")
print(f"K-averaging: {K_AVERAGING}" + (f" (k={K_VALUE})" if K_AVERAGING else ""))
if K_AVERAGING:
    print(f"Batch eval: {RUN_BATCH_EVAL}" + (f" (n={KAVG_BATCH_SIZE})" if RUN_BATCH_EVAL else ""))
print(f"Run A/B/C batch eval: {RUN_ABC_BATCH_EVAL}" + (f" (runs={sorted(ABC_BATCH_RUNS)})" if RUN_ABC_BATCH_EVAL else ""))

## 3. Network Architecture

`build_network()` is a factory that dispatches on `ARCHITECTURE`, so every place in the notebook that needs a fresh network calls this one function instead of instantiating `conv_model` directly. When you add a new architecture, implement its class here and add one `elif` branch — no other cell needs to change.

It also calls `set_seed()` internally before construction, so weight initialization is reproducible and — critically — identical across `Z_SOURCE` toggles for the same `ARCHITECTURE` and `SEED` (isolating the z-source comparison from random-init noise).

In [ ]:
# --- ConvDecoder architecture ---

def add_module(self, module):
    self.add_module(str(len(self) + 1), module)
torch.nn.Module.add = add_module

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        hidden = max(channels // reduction, 4)
        self.fc1 = nn.Linear(channels, hidden)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(hidden, channels)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.relu(self.fc1(s))
        s = self.sigmoid(self.fc2(s))
        return x * s.view(b, c, 1, 1)

class conv_model(nn.Module):
    def __init__(self, num_layers, num_channels, num_output_channels, out_size, in_size, use_se=False):
        super(conv_model, self).__init__()
        kernel_size = 3
        strides = [1] * (num_layers - 1)
        scale_x = (out_size[0] / in_size[0]) ** (1. / (num_layers - 1))
        scale_y = (out_size[1] / in_size[1]) ** (1. / (num_layers - 1))
        hidden_size = [(int(np.ceil(scale_x**n * in_size[0])),
                        int(np.ceil(scale_y**n * in_size[1])))
                       for n in range(1, (num_layers - 1))] + [out_size]

        self.net = nn.Sequential()
        for i in range(num_layers - 1):
            self.net.add(nn.Upsample(size=hidden_size[i], mode='nearest'))
            conv = nn.Conv2d(num_channels, num_channels, kernel_size, strides[i],
                              padding=(kernel_size - 1) // 2, bias=True)
            self.net.add(conv)
            self.net.add(nn.ReLU())
            self.net.add(nn.BatchNorm2d(num_channels, affine=True))
            if use_se:
                self.net.add(SEBlock(num_channels))
        self.net.add(nn.Conv2d(num_channels, num_channels, kernel_size, strides[i],
                                padding=(kernel_size - 1) // 2, bias=True))
        self.net.add(nn.ReLU())
        self.net.add(nn.BatchNorm2d(num_channels, affine=True))
        if use_se:
            self.net.add(SEBlock(num_channels))
        self.net.add(nn.Conv2d(num_channels, num_output_channels, 1, 1, padding=0, bias=True))

    def forward(self, x, scale_out=1):
        return self.net(x) * scale_out


def build_network(shape_info, seed=SEED):
    set_seed(seed)
    if ARCHITECTURE == "convdecoder":
        return conv_model(num_layers, num_channels,
                           shape_info['output_depth'], shape_info['out_size'], in_size).type(dtype)
    elif ARCHITECTURE == "convdecoder_se":
        return conv_model(num_layers, num_channels,
                           shape_info['output_depth'], shape_info['out_size'], in_size, use_se=True).type(dtype)
    else:
        raise ValueError(f"No builder registered for architecture '{ARCHITECTURE}'. "
                          f"Add an elif branch here when you implement it.")

## 4. Input Code (z) Source

`get_z(u['zf_complex_cropped'])` is the single dispatch point controlled by `Z_SOURCE`:
- `"uniform"` → the original fixed Uniform[0,1] ConvDecoder input.
- `"mri_vae"` → a deterministic latent derived from the current image's complex zero-filled reconstruction using Microsoft's pretrained `mri-autoencoder-v0.1`.

The MRI VAE is downloaded and loaded only for `Z_SOURCE == "mri_vae"`. It remains frozen. The posterior mean is used instead of sampling, then its four latent channels are deterministically expanded and resized to the unchanged ConvDecoder input shape `[1, 256, 8, 4]`. `ARCHITECTURE` remains an independent toggle, so all four Uniform/MRI-VAE × original/SE combinations remain available.


In [ ]:
if Z_SOURCE == "mri_vae":
    !pip install -q "diffusers>=0.24.0" "huggingface_hub>=0.20.0" safetensors
    from diffusers.models import AutoencoderKL

    MRI_VAE_MODEL_ID = "microsoft/mri-autoencoder-v0.1"
    _vae = AutoencoderKL.from_pretrained(
        MRI_VAE_MODEL_ID,
        torch_dtype=torch.float32,
    ).to("cuda").eval()
    for p in _vae.parameters():
        p.requires_grad_(False)
    assert not _vae.training and not any(p.requires_grad for p in _vae.parameters())
    print(f"Loaded frozen pretrained MRI VAE: {MRI_VAE_MODEL_ID}")

    @torch.no_grad()
    def mri_vae_encode_to_z(complex_image_2d, in_size=in_size):
        """Encode one complex zero-filled MRI image into ConvDecoder's fixed input z.

        Microsoft preprocessing: two channels [real, imag], normalized by the 99.5th
        percentile of complex magnitude. The posterior mean is deterministic. The VAE has
        4 latent channels; repeat (not random projection) expands them to num_channels, and
        spatial interpolation preserves the notebook's existing ConvDecoder input shape.
        """
        image = np.asarray(complex_image_2d)
        if image.ndim != 2 or not np.iscomplexobj(image):
            raise ValueError(
                f"MRI VAE expects a 2D complex image; got shape={image.shape}, dtype={image.dtype}."
            )
        scale = float(np.percentile(np.abs(image), 99.5))
        if not np.isfinite(scale) or scale <= 0:
            raise ValueError(f"MRI VAE input has invalid 99.5th-percentile magnitude: {scale!r}")

        normalized = image / scale
        two_channel = np.stack((normalized.real, normalized.imag), axis=0).astype(np.float32)
        x = torch.from_numpy(two_channel)[None].to(device="cuda", dtype=torch.float32)

        # The published model was trained at 256x256. Resize real/imaginary channels together.
        x = F.interpolate(x, size=(256, 256), mode="bilinear", align_corners=False)
        posterior = _vae.encode(x).latent_dist
        latent = posterior.mean  # deterministic image-related code; do not sample during A/B tests

        # Deterministic channel expansion: every latent channel is represented equally.
        repeats = int(np.ceil(num_channels / latent.shape[1]))
        z = latent.repeat(1, repeats, 1, 1)[:, :num_channels]
        z = F.interpolate(z, size=tuple(in_size), mode="bilinear", align_corners=False)

        # Match the numerical range of the original fixed Uniform[0,1] input without destroying
        # spatial/image dependence. This also makes initial-output scaling comparisons well behaved.
        z_min, z_max = z.amin(), z.amax()
        z = (z - z_min) / (z_max - z_min).clamp_min(1e-8)
        if not torch.isfinite(z).all():
            raise ValueError("MRI VAE produced a non-finite ConvDecoder input z.")
        return torch.autograd.Variable(z).type(dtype)
else:
    def mri_vae_encode_to_z(*args, **kwargs):
        raise RuntimeError(
            "Z_SOURCE is 'uniform' so the MRI VAE was never loaded. "
            "Set Z_SOURCE = 'mri_vae' in the Config cell and re-run from there if intended."
        )


def get_z(zf_complex_cropped):
    """Single dispatch point for the fixed input code z."""
    if Z_SOURCE == "uniform":
        return None
    if Z_SOURCE == "mri_vae":
        return mri_vae_encode_to_z(zf_complex_cropped)
    raise ValueError(f"Unknown Z_SOURCE: {Z_SOURCE!r}")


## 5. Helper Functions

Forward model, scale-factor estimation, normalization, and the metrics used in the final results table.

In [ ]:
def forwardm(img, mask):
    '''Multicoil forward model: image (real/imag channels per coil) -> masked k-space.'''
    mask_t = np_to_var(mask)[0].type(dtype)
    s = img.shape
    ns = int(s[1] / 2)
    fimg = Variable(torch.zeros((s[0], ns, s[2], s[3], 2))).type(dtype)
    for i in range(ns):
        fimg[0, i, :, :, 0] = img[0, 2 * i, :, :]
        fimg[0, i, :, :, 1] = img[0, 2 * i + 1, :, :]
    Fimg = fft2(fimg)
    for i in range(ns):
        Fimg[0, i, :, :, 0] *= mask_t
        Fimg[0, i, :, :, 1] *= mask_t
    return Fimg


def channels2imgs_complex(out):
    '''Like channels2imgs(), but keeps phase (returns complex per-coil images) instead of
    collapsing to magnitude -- ESPIRiT/Roemer combination needs each coil's relative phase to
    combine correctly.'''
    sh = out.shape
    chs = int(sh[0] / 2)
    imgs = np.zeros((chs, sh[1], sh[2]), dtype=np.complex64)
    for i in range(chs):
        imgs[i] = out[2 * i] + 1j * out[2 * i + 1]
    return imgs


def safe_calib_width(masked_kspace, cent=0.07, margin=0.9, min_width=8):
    '''Compute a calib_width for EspiritCalib that's guaranteed to stay inside THIS image's
    actual fully-sampled ACS region, instead of assuming one fixed size for every image. Per
    get_mask()/MaskFunc, the guaranteed-fully-sampled low-frequency block is round(num_cols *
    cent) columns wide, where num_cols is this image's own phase-encode-dimension width (which
    can vary across scans with different matrix sizes) -- NOT some dataset-wide constant. Using
    a fixed calib_width that happens to exceed that width for a given image would make
    EspiritCalib calibrate from a region that includes randomly-undersampled columns, corrupting
    the maps rather than improving them, so this always caps calib_width at that per-image bound.

    margin shrinks the true ACS width by a small buffer (rounding/off-by-one safety); min_width
    is just a quality heads-up threshold, not a hard floor -- safety (never exceeding the true
    ACS width) always takes priority over hitting it.'''
    num_cols = masked_kspace.shape[-2]
    acs_width = int(round(num_cols * cent))
    calib_width = min(int(acs_width * margin), acs_width)
    calib_width = max(calib_width, 1)
    if calib_width < min_width:
        print(f"  WARNING: this image's guaranteed ACS width is only {acs_width} columns "
              f"(num_cols={num_cols}, cent={cent}) -- using calib_width={calib_width}, below "
              f"the usual quality floor of {min_width}. ESPIRiT maps may be noisy for this image.")
    return calib_width


class EspiritCalibrationError(RuntimeError):
    '''Raised when ESPIRiT calibration can't find usable coil-sensitivity support for an
    image, even after retrying with a larger calibration window. Proceeding to fit against this
    image's (near-zero) maps would divide by ~0 in get_scale_factor and corrupt the fit into
    NaN/Inf from iteration 0 -- callers should catch this and skip the image (see the batch-eval
    loops in Sections 12.5.2/13, which already skip unreadable files the same way) rather than
    let a bad calibration silently poison a fit.'''
    pass


def espirit_maps(masked_kspace, calib_width=None, cent=0.07, min_support=0.05):
    '''Estimate ESPIRiT coil sensitivity maps (sigpy.mri.app.EspiritCalib) from the
    always-fully-sampled, low-frequency center (ACS) of the measured k-space. masked_kspace is
    the (num_coils, H, W, 2) real/imag torch tensor produced by apply_mask(); EspiritCalib crops
    its own calib_width x calib_width calibration window from the array center internally, so
    the full (zero-padded outside the ACS) masked k-space can be passed directly.

    calib_width=None (the default) computes a safe, per-image value via safe_calib_width() above,
    rather than assuming one fixed size is safe for every image in the dataset. If that first
    attempt yields low sensitivity support (most images won't), this retries once with the full
    ACS width (no safety margin -- the largest calibration window that's still guaranteed not to
    include any undersampled columns). If support is still low after that, raises
    EspiritCalibrationError rather than returning a map that would silently corrupt everything
    downstream -- a genuinely weak/noisy ACS region for this one scan, not something a bigger
    calib_width can always fix.'''
    def _calibrate(width):
        ksp_np = masked_kspace.detach().cpu().numpy()
        ksp_complex = (ksp_np[..., 0] + 1j * ksp_np[..., 1]).astype(np.complex64)
        m = mri.app.EspiritCalib(ksp_complex, calib_width=int(width), show_pbar=False).run()
        if not np.all(np.isfinite(m)):
            return m, 0.0
        energy = np.sum(np.abs(m) ** 2, axis=0)
        peak = float(np.max(energy)) if energy.size else 0.0
        support = float(np.mean(energy > max(1e-12, 1e-6 * peak))) if peak > 0 else 0.0
        return m, support

    if calib_width is None:
        calib_width = safe_calib_width(masked_kspace, cent=cent)
    num_cols = masked_kspace.shape[-2]
    max_calib_width = max(1, int(round(num_cols * cent)))
    # Increasing calib_width is only legal up to the contiguous fully-sampled ACS width.
    widths = sorted(set([int(calib_width),
                         min(max_calib_width, int(round(0.95 * max_calib_width))),
                         max_calib_width]))
    mps, support_fraction = None, 0.0
    for attempt, width in enumerate(widths, 1):
        mps_try, support_try = _calibrate(width)
        if attempt > 1:
            print(f"  ESPIRiT retry {attempt}/{len(widths)}: calib_width={width}, "
                  f"support={support_try:.2%}")
        if support_try > support_fraction:
            mps, support_fraction = mps_try, support_try
        if support_try >= min_support:
            mps, support_fraction = mps_try, support_try
            break

    if support_fraction < min_support:
        raise EspiritCalibrationError(
            f"ESPIRiT calibration found almost no sensitivity support ({support_fraction:.2%} "
            f"of pixels nonzero) even after retrying with the full ACS width. This image's ACS "
            f"region is likely too weak/noisy for reliable calibration."
        )
    return mps


def espirit_combine(coil_images_complex, mps, support_thresh=0.05):
    '''Roemer/ESPIRiT sensitivity-weighted coil combination in image space.
    coil_images_complex and mps are both (num_coils, H, W) complex arrays; returns the complex
    combined image (H, W). Take np.abs(...) of the result for a magnitude image.

    EspiritCalib deliberately zeroes mps outside its coil-sensitivity support (background/air
    regions), so den = sum(|mps|^2) goes to ~0 there. Dividing by a fixed epsilon in that region
    doesn't fix this -- it turns whatever tiny numerator noise exists into large spurious values,
    since dividing by something close to zero massively amplifies it. Any pixel whose support
    falls below support_thresh (as a fraction of this image's
    peak support) is treated as "no reliable sensitivity data" and combined to exactly 0, instead
    of being divided out into noise.'''
    num = np.sum(coil_images_complex * np.conj(mps), axis=0)
    den = np.sum(np.abs(mps) ** 2, axis=0)
    support_mask = den > (support_thresh * den.max())
    combined = np.zeros_like(num)
    combined[support_mask] = num[support_mask] / den[support_mask]
    return combined


def make_espirit_reference(full_kspace, mps, crop_shape=(320, 320)):
    '''Build the fully sampled ESPIRiT reference using the same sensitivity maps used for
    the undersampled reconstruction. full_kspace is complex with shape (coils, H, W).'''
    full_kspace = np.asarray(full_kspace)
    full_kspace_ri = torch.from_numpy(
        np.stack((full_kspace.real, full_kspace.imag), axis=-1)
    )
    full_coils_ri = ifft2(full_kspace_ri).cpu().numpy()
    full_coils = full_coils_ri[..., 0] + 1j * full_coils_ri[..., 1]
    reference = np.abs(espirit_combine(full_coils, mps))
    if crop_shape is not None:
        reference = crop_center(reference, crop_shape[0], crop_shape[1])
    return reference


def get_scale_factor(net, num_channels, in_size, masked_kspace, mps=None, ni=None):
    """Match initial prediction and target RMS over acquired multicoil k-space samples.

    This deliberately does not use ESPIRiT: scaling belongs to the same domain as the
    data-consistency loss, and therefore remains stable even when map support is small.
    """
    if ni is None:
        shape = [1, num_channels, in_size[0], in_size[1]]
        ni = Variable(torch.zeros(shape)).type(dtype)
        ni.data.uniform_()
    with torch.no_grad():
        pred = forwardm(net(ni.type(dtype)), np.ones(masked_kspace.shape[-3:-1], dtype=np.float32))[0]
        target = masked_kspace.to(device=pred.device, dtype=pred.dtype)
        acquired = torch.any(target != 0, dim=-1, keepdim=True)
        pred_vals = pred.masked_select(acquired.expand_as(pred))
        target_vals = target.masked_select(acquired.expand_as(target))
        pred_rms = torch.sqrt(torch.mean(pred_vals.float().square()))
        target_rms = torch.sqrt(torch.mean(target_vals.float().square()))
        eps = torch.finfo(torch.float32).eps
        s = (pred_rms / target_rms.clamp_min(eps)).item()
    if not np.isfinite(s) or s <= 0:
        raise EspiritCalibrationError(
            f"Non-finite pre-fit k-space scale ({s!r}); pred_rms={pred_rms.item():.6g}, "
            f"target_rms={target_rms.item():.6g}. Skipping this image safely."
        )
    return s, ni


def normalize(img):
    '''Scale an image to [0, max=1] range so different reconstructions are comparable.'''
    return img / img.max()


def nmse(gt, pred):
    return np.linalg.norm(gt - pred) ** 2 / np.linalg.norm(gt) ** 2


def psnr(gt, pred):
    return psnr_fn(gt, pred, data_range=gt.max())


def ssim(gt, pred):
    return ssim_fn(gt, pred, data_range=gt.max())


def ms_ssim(gt, pred):
    gt_t = torch.tensor(gt, dtype=torch.float32)[None, None]
    pred_t = torch.tensor(pred, dtype=torch.float32)[None, None]
    return pytorch_msssim.ms_ssim(gt_t, pred_t, data_range=gt.max()).item()


def vif(gt, pred):
    return vifp(gt, pred)


def hfen(gt, pred, sigma=1.5):
    '''High-Frequency Error Norm — sensitive to loss of fine detail/edges. Lower is better.'''
    gt_log = gaussian_laplace(gt.astype(np.float64), sigma=sigma)
    pred_log = gaussian_laplace(pred.astype(np.float64), sigma=sigma)
    return np.linalg.norm(gt_log - pred_log) / np.linalg.norm(gt_log)


def compute_all_metrics(gt, pred):
    '''gt and pred should already be normalize()-d before calling this.'''
    return {
        'PSNR': psnr(gt, pred),
        'SSIM': ssim(gt, pred),
        'MS-SSIM': ms_ssim(gt, pred),
        'VIF': vif(gt, pred),
        'NMSE': nmse(gt, pred),
        'HFEN': hfen(gt, pred),
    }


def build_undersampled(slice_ksp_torchtensor, slice_ksp, net_for_scale, factor=4, cent=0.07, calib_width=None):
    '''One-stop helper: mask -> apply -> ESPIRiT calibrate -> scale -> measurement + zero-filled
    image. Returns a dict with everything needed for fitting and visualization.
    NOTE: call set_seed(SEED) immediately before this if you need the mask to be
    reproducible/matched across a Z_SOURCE or ARCHITECTURE toggle (see Sections 6/7).'''
    mask, mask1d, mask2d = get_mask(slice_ksp_torchtensor, slice_ksp, factor=factor, cent=cent)
    masked_kspace, _ = apply_mask(slice_ksp_torchtensor, mask=mask)

    # ESPIRiT sensitivity maps, estimated once from this measurement's ACS region and reused
    # for scaling, the zero-filled preview, and every reconstruct() call on this image, so coil
    # combination is identical (same maps) everywhere it's used for a given image.
    mps = espirit_maps(masked_kspace, calib_width=calib_width, cent=cent)

    scaling_factor, ni = get_scale_factor(net_for_scale, num_channels, in_size, masked_kspace, mps, ni=None)
    masked_kspace_scaled = masked_kspace * scaling_factor
    unders_measurement = Variable(masked_kspace_scaled[None, :]).type(dtype)

    orig_tt = ifft2(masked_kspace)
    orig_np = orig_tt.cpu().numpy()
    orig_imgs_complex = orig_np[..., 0] + 1j * orig_np[..., 1]
    zf_complex = espirit_combine(orig_imgs_complex, mps)
    zf_complex_cropped = crop_center(zf_complex, 320, 320)
    zf_img = np.abs(zf_complex)
    zf_img_cropped = np.abs(zf_complex_cropped)
    gt_espirit = make_espirit_reference(slice_ksp, mps)

    return {
        'mask': mask, 'mask1d': mask1d, 'mask2d': mask2d,
        'masked_kspace': masked_kspace, 'scaling_factor': scaling_factor, 'ni': ni,
        'unders_measurement': unders_measurement, 'zf_img_cropped': zf_img_cropped,
        'zf_complex_cropped': zf_complex_cropped,
        'mps': mps, 'gt_espirit': gt_espirit,
    }


def reconstruct(net, ni, mps):
    '''Run a fitted network forward and return the cropped, ESPIRiT/Roemer-combined image.
    mps are the coil sensitivity maps for this image, from build_undersampled()'s 'mps' entry.'''
    out = net(ni.type(dtype))
    out_chs = out.data.cpu().numpy()[0]
    out_imgs_complex = channels2imgs_complex(out_chs)
    rec = np.abs(espirit_combine(out_imgs_complex, mps))
    return crop_center(rec, 320, 320)


DIFF_VMAX = 0.15
# Cap for the difference-map colormap. imshow(..., cmap='hot') with no vmin/vmax auto-scales
# to the diff array's own min/max, which stretches the MRI background's noise floor (Rician
# noise present even in air/background regions of a magnitude image) to look just as "hot" as
# genuine reconstruction error around the anatomy. Fixing vmin=0/vmax=DIFF_VMAX keeps the color
# scale consistent across figures and keeps background noise from drowning out real error.

def show_difference(gt, recon, title, vmax=DIFF_VMAX):
    gt_n = normalize(gt)
    recon_n = normalize(recon)
    diff = np.abs(gt_n - recon_n)
    # np.flipud(...) only for display -- these images come out of the forward model
    # upside-down relative to normal radiological viewing, so every panel is flipped
    # right before imshow. Metrics/diff values above are computed on the un-flipped
    # arrays, so this has no effect on any reported number.
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(np.flipud(gt), cmap='gray'); axes[0].set_title('Ground Truth'); axes[0].axis('off')
    axes[1].imshow(np.flipud(recon), cmap='gray'); axes[1].set_title(title); axes[1].axis('off')
    axes[2].imshow(np.flipud(diff), cmap='gray', vmin=0, vmax=vmax); axes[2].set_title('Difference'); axes[2].axis('off')
    plt.show()
    print(f"Mean abs difference: {diff.mean():.4f}   Max abs difference: {diff.max():.4f}")


def show_full_comparison(gt, zf, recon, recon_title, vmax=DIFF_VMAX):
    gt_n = normalize(gt)
    recon_n = normalize(recon)
    diff = np.abs(gt_n - recon_n)

    # np.flipud(...) only for display -- see note in show_difference() above.
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    axes[0].imshow(np.flipud(gt), cmap='gray'); axes[0].set_title('Ground Truth'); axes[0].axis('off')
    axes[1].imshow(np.flipud(zf), cmap='gray'); axes[1].set_title('Zero-filled (Undersampled)'); axes[1].axis('off')
    axes[2].imshow(np.flipud(recon), cmap='gray'); axes[2].set_title(recon_title); axes[2].axis('off')
    axes[3].imshow(np.flipud(diff), cmap='gray', vmin=0, vmax=vmax); axes[3].set_title('Difference (GT vs Reconstruction)'); axes[3].axis('off')
    plt.show()
    print(f"Mean abs difference: {diff.mean():.4f}   Max abs difference: {diff.max():.4f}")

### 5.1 Custom Loss Function & Regularization

`demo_helper.fit_multicoil.fit()` (imported in Section 1) hardcodes `torch.nn.MSELoss()` as the data-fidelity loss and has no regularization term. That file lives in the cloned repo, which gets re-cloned fresh every session, so editing it on disk wouldn't persist. Instead, this cell defines a replacement `fit()` with the identical signature and checkpointing behavior, but with the loss swapped out based on `LOSS_TYPE`/`HUBER_DELTA` and an optional Total Variation term weighted by `TV_WEIGHT` (Section 2). Defining it here makes Python use this version for every `fit(...)` call later in the notebook, since it's the same name assigned more recently.

TV regularization is computed directly on the network's raw output tensor (the per-coil real/imaginary channels, before the forward model/mask are applied) — it penalizes large jumps between neighboring pixels in each channel, encouraging a piecewise-smooth reconstruction. With `TV_WEIGHT = 0` (default-equivalent), this reduces to plain MSE fitting, identical to the original `fit()`.

In [ ]:
from tqdm.auto import tqdm

class ReconstructionDivergedError(RuntimeError):
    """Raised before saving/using a fit when output, loss, gradients, or weights become non-finite."""
    pass

def fit(net, img_noisy_var, num_channels, net_input, apply_f, mask,
        mask1d=None, scaling_factor=None, num_iter=5000, LR=0.01,
        checkpoint_dir=None, checkpoint_every=1000, dtype=torch.cuda.FloatTensor):
    '''Drop-in replacement for demo_helper.fit_multicoil.fit — same signature and checkpointing
    behavior, extended with a configurable data-fidelity loss (LOSS_TYPE/HUBER_DELTA) and an
    optional Total Variation regularizer (TV_WEIGHT), both read from the Section 2 config.'''
    net_input = net_input.type(dtype)
    p = [x for x in net.parameters()]
    mse_wrt_noisy = np.zeros(num_iter)

    criterion = torch.nn.MSELoss() if LOSS_TYPE == "mse" else torch.nn.HuberLoss(delta=HUBER_DELTA)

    def tv_loss(img):
        '''Total variation on the raw per-coil real/imag channel tensor: mean absolute difference
        between horizontally/vertically neighboring pixels, summed. Encourages piecewise-smooth
        output; penalizes noisy/high-frequency artifacts more than genuine structure.'''
        dh = torch.abs(img[:, :, 1:, :] - img[:, :, :-1, :]).mean()
        dw = torch.abs(img[:, :, :, 1:] - img[:, :, :, :-1]).mean()
        return dh + dw

    print(f"optimize with adam {LR}  |  loss: {LOSS_TYPE}"
          f"{f' (delta={HUBER_DELTA})' if LOSS_TYPE == 'huber' else ''}"
          f"  |  TV weight: {TV_WEIGHT}")
    optimizer = torch.optim.Adam(p, lr=LR)

    best_net = copy.deepcopy(net)
    best_mse = 1000000.0

    pbar = tqdm(range(num_iter), desc="Fitting", unit="it")
    for i in pbar:
        def closure():
            optimizer.zero_grad()
            out = net(net_input.type(dtype))
            if not torch.isfinite(out).all():
                raise ReconstructionDivergedError(f"non-finite network output at iteration {i}")
            prediction = apply_f(out, mask)
            data_term = criterion(prediction, img_noisy_var)
            tv_term = tv_loss(out) if TV_WEIGHT > 0 else out.new_zeros(())
            loss = data_term + TV_WEIGHT * tv_term
            if not torch.isfinite(loss):
                raise ReconstructionDivergedError(
                    f"non-finite loss at iteration {i} (data={data_term.item()}, tv={tv_term.item()})")
            loss.backward()
            if any(p.grad is not None and not torch.isfinite(p.grad).all() for p in net.parameters()):
                raise ReconstructionDivergedError(f"non-finite gradient at iteration {i}")
            mse_wrt_noisy[i] = loss.detach().cpu().item()
            return loss

        loss = optimizer.step(closure)
        if any(not torch.isfinite(p).all() for p in net.parameters()):
            raise ReconstructionDivergedError(f"non-finite model weight after iteration {i}")
        pbar.set_postfix(loss=f"{loss.detach().item():.6f}")

        if best_mse > 1.005 * loss.data:
            best_mse = loss.data
            best_net = copy.deepcopy(net)

        if checkpoint_dir is not None and (i % checkpoint_every == 0 or i == num_iter - 1):
            os.makedirs(checkpoint_dir, exist_ok=True)
            torch.save({
                "iteration": i,
                "model_state_dict": best_net.state_dict(),
                "net_input": net_input,
                "mask": mask,
                "mask1d": mask1d,
                "scaling_factor": scaling_factor,
            }, os.path.join(checkpoint_dir, "checkpoint.pt"))

    net = best_net
    return mse_wrt_noisy, net

## 6. Load Data Samples

Pulls a handful of multicoil knee `.h5` files from Drive. Each sample keeps the raw multicoil k-space and its torch tensor form. Fully sampled references are generated later with ESPIRiT, using the same ACS-derived maps as each undersampled reconstruction.


In [ ]:
folder = "/content/drive/MyDrive/Projects/knee_multicoil_val/multicoil_val"

t0 = time.time()
all_files = sorted([f for f in os.listdir(folder) if f.endswith('.h5')])
print(f"Found {len(all_files)} files ({time.time()-t0:.1f}s)")

# Hand-picked non-fat-suppressed (CORPD_FBK) files confirmed during exploration
selected_files = ['file1000031.h5', 'file1000033.h5', 'file1000041.h5',
                   'file1000071.h5', 'file1000073.h5']

samples = []
for fname in selected_files:
    filepath = os.path.join(folder, fname)
    try:
        with h5py.File(filepath, 'r') as f:
            print(fname, "—", dict(f.attrs))
            slicenu = f["kspace"].shape[0] // 2
            slice_ksp = f['kspace'][slicenu]
            data = slice_ksp.copy()
            data = np.stack((data.real, data.imag), axis=-1)
            slice_ksp_torchtensor = torch.from_numpy(data)
            samples.append({
                'filename': fname,
                'slice_ksp': slice_ksp,
                'slice_ksp_torchtensor': slice_ksp_torchtensor,
            })
    except OSError as e:
        print(f"  SKIPPED {fname}: {e}")
        continue

print(f"\nLoaded {len(samples)} samples successfully (out of {len(selected_files)} attempted).")

### 6.1 Visualize undersampling masks

Draws and displays the actual random Cartesian undersampling mask for a few
of the loaded demo images, using the same `get_mask()` / `build_undersampled()`
helpers used everywhere else — this is exactly what fitting sees, not a
re-derived approximation. Note: the mask is redrawn per call (see the note
in Section 2 about `MaskFunc`'s RNG not being tied to `SEED`), so re-running
this cell can show a different mask pattern each time for the same image.

In [ ]:
# # ============================================================
# # Visualize undersampling masks
# # ============================================================
# # Shows the random Cartesian mask (fully-sampled center + random remaining
# # columns, ~4x average acceleration) for a few already-loaded demo images.

# def _to_numpy(x):
#     return x.cpu().numpy() if hasattr(x, 'cpu') else np.asarray(x)

# mask_demo_filenames = [s['filename'] for s in samples[:4]]  # first 4 loaded demo images

# fig, axes = plt.subplots(2, len(mask_demo_filenames), figsize=(4 * len(mask_demo_filenames), 6))

# for i, fname in enumerate(mask_demo_filenames):
#     sample = next(s for s in samples if s['filename'] == fname)
#     slice_ksp_torchtensor = sample['slice_ksp_torchtensor']
#     slice_ksp = sample['slice_ksp']

#     net_for_mask = build_network({
#         'output_depth': slice_ksp_torchtensor.numpy().shape[0] * 2,
#         'out_size': slice_ksp_torchtensor.numpy().shape[1:-1],
#     })
#     set_seed(SEED)  # matches how the mask is drawn everywhere else (Sections 7/8/12)
#     u = build_undersampled(slice_ksp_torchtensor, slice_ksp, net_for_mask)
#     mask1d, mask2d = _to_numpy(u['mask1d']), _to_numpy(u['mask2d'])

#     realized_accel = mask1d.size / mask1d.sum()

#     # Top row: 1D pattern -- which phase-encode columns were kept
#     axes[0, i].plot(mask1d.flatten(), linewidth=0.8, color='#13293D')
#     axes[0, i].set_title(f'{fname}\n{realized_accel:.2f}x realized accel.', fontsize=10)
#     axes[0, i].set_xlabel('phase-encode column')
#     axes[0, i].set_yticks([0, 1])
#     axes[0, i].set_ylim(-0.1, 1.1)

#     # Bottom row: full 2D k-space mask
#     axes[1, i].imshow(mask2d, cmap='gray', aspect='auto')
#     axes[1, i].set_title('2D k-space mask', fontsize=10)
#     axes[1, i].axis('off')

# plt.suptitle('Undersampling masks (target: 4x acceleration, 7% fully-sampled center)', y=1.02)
# plt.tight_layout()
# plt.show()

### 6.5 (optional, already run) Diagnostic probe + Optuna search for `HUBER_DELTA` / `TV_WEIGHT`

The code cell below is **commented out on purpose** — it's a *record of a completed hyperparameter search*, not something `Run All` should re-execute automatically (a full run costs ~25 short fits across 5 images). It:

1. draws 5 tuning images at random, excluded from the demo images loaded in Section 6 -- but **not yet** excluded from the eval/reference pools Sections 12.5.1/13 build later, so add the printed `TUNING_FILENAMES` to `KAVG_BATCH_MANUAL_EXCLUDE` (Section 2) if you re-run this before running those sections;
2. runs a short plain-MSE probe fit to read off a sane residual scale and center the search range on it;
3. searches `(HUBER_DELTA, TV_WEIGHT)` with Optuna (25 trials × 300-iteration partial fits per trial, averaged over the 5 tuning images), maximizing `PSNR - 10*HFEN`.


To retune from scratch: uncomment the cell, adjust `N_TUNING_IMAGES` / `N_OPTUNA_TRIALS` / `TUNING_PROBE_ITERS` if needed, run it, copy the resulting `HUBER_DELTA` / `TV_WEIGHT` into Section 2, then re-comment this cell before your next `Run All` so it doesn't re-run unintentionally.


In [ ]:
# # ============================================================
# # Diagnostic probe + Optuna search for HUBER_DELTA / TV_WEIGHT
# # ============================================================
# # It needs `samples`, `all_files`, `folder` from Section 6, and `build_network`,
# # `get_z`, `get_scale_factor`, `build_undersampled`, `forwardm`, `fit`, `reconstruct`,
# # `normalize`, `psnr`, `hfen`, `set_seed` from Sections 3-5.
# #
# # TUNING_FILENAMES are drawn at random from the full file list, excluding only the 5
# # hand-picked demo files already loaded into `samples` (Sections 7/8/12 reuse those).
# # They are NOT yet excluded from the eval/reference pools Sections 12.5.1 and 13 build
# # later -- add TUNING_FILENAMES to KAVG_BATCH_MANUAL_EXCLUDE (Section 2) before running
# # those, so the tuning set and the evaluation set never overlap.

# !pip install optuna -q
# import optuna

# N_TUNING_IMAGES = 5          # small, cheap set -- just enough to average out per-image noise
# TUNING_PROBE_ITERS = 300     # short partial fit per trial, not a full 10,000-iter fit
# N_OPTUNA_TRIALS = 25

# random.seed(SEED)
# _tuning_candidates = [f for f in all_files if f not in {s['filename'] for s in samples}]
# TUNING_FILENAMES = random.sample(_tuning_candidates, N_TUNING_IMAGES)
# print(f"Tuning images ({len(TUNING_FILENAMES)}): {TUNING_FILENAMES}")
# print("NOTE: add these filenames to KAVG_BATCH_MANUAL_EXCLUDE (Section 2) before running "
#       "Section 12.5.1 / 13, so the tuning set and the evaluation set never overlap.")

# # --- Step 1: diagnostic probe -- one short MSE fit, just to read off a sane
# # residual scale before searching. Not itself part of the Optuna objective.
# _probe_fname = TUNING_FILENAMES[0]
# with h5py.File(os.path.join(folder, _probe_fname), 'r') as f:
#     slicenu_p = f["kspace"].shape[0] // 2
#     slice_ksp_p = f['kspace'][slicenu_p]
#     data_p = np.stack((slice_ksp_p.real, slice_ksp_p.imag), axis=-1)
# ksp_tt_p = torch.from_numpy(data_p)
# output_depth_p = ksp_tt_p.numpy().shape[0] * 2
# out_size_p = ksp_tt_p.numpy().shape[1:-1]

# _old_loss, _old_delta, _old_tv = LOSS_TYPE, HUBER_DELTA, TV_WEIGHT
# LOSS_TYPE, TV_WEIGHT = "mse", 0.0   # probe with plain MSE, no TV, to see the raw residual scale

# net_p = build_network({'output_depth': output_depth_p, 'out_size': out_size_p})
# set_seed(SEED)
# u_p = build_undersampled(ksp_tt_p, slice_ksp_p, net_p)
# ni_p = get_z(u_p['zf_complex_cropped'])
# scaling_factor_p, ni_p = get_scale_factor(net_p, num_channels, in_size, u_p['masked_kspace'], u_p['mps'], ni=ni_p)
# unders_measurement_p = Variable((u_p['masked_kspace'] * scaling_factor_p)[None, :]).type(dtype)

# mse_probe, _ = fit(
#     net=net_p, img_noisy_var=unders_measurement_p, num_channels=num_channels,
#     net_input=ni_p, apply_f=forwardm, mask=u_p['mask2d'], mask1d=u_p['mask1d'],
#     scaling_factor=scaling_factor_p, num_iter=TUNING_PROBE_ITERS, LR=0.01, checkpoint_dir=None,
# )
# probe_residual_scale = float(np.sqrt(mse_probe[-1]))  # rough per-pixel residual magnitude late in the probe
# print(f"Diagnostic probe residual scale (~sqrt(final MSE)): {probe_residual_scale:.4f}")

# LOSS_TYPE, HUBER_DELTA, TV_WEIGHT = _old_loss, _old_delta, _old_tv  # restore config

# # Search ranges centered on the probe's scale: delta near/below the residual scale is
# # where Huber starts diverging from MSE; TV_WEIGHT spans a few orders of magnitude
# # since its right scale depends on image statistics, not residual statistics.
# DELTA_LOW, DELTA_HIGH = 0.05 * probe_residual_scale, 2.0 * probe_residual_scale
# TV_LOW, TV_HIGH = 1e-6, 1e-2

# # --- Step 2: preload every tuning image's (mask, measurement, ground truth) ONCE so
# # every trial reuses the exact same undersampling, instead of drawing a fresh random
# # mask per trial (which would add mask noise on top of hyperparameter noise).
# _tuning_data = []
# for fname in TUNING_FILENAMES:
#     with h5py.File(os.path.join(folder, fname), 'r') as f:
#         slicenu_i = f["kspace"].shape[0] // 2
#         slice_ksp_i = f['kspace'][slicenu_i]
#         data_i = np.stack((slice_ksp_i.real, slice_ksp_i.imag), axis=-1)
#     ksp_tt_i = torch.from_numpy(data_i)
#     output_depth_i = ksp_tt_i.numpy().shape[0] * 2
#     out_size_i = ksp_tt_i.numpy().shape[1:-1]
#     net_for_mask_i = build_network({'output_depth': output_depth_i, 'out_size': out_size_i})
#     set_seed(SEED)
#     u_i = build_undersampled(ksp_tt_i, slice_ksp_i, net_for_mask_i)
#     gt_i = normalize(u_i['gt_espirit'].astype(np.float64))
#     _tuning_data.append({
#         'filename': fname, 'output_depth': output_depth_i, 'out_size': out_size_i,
#         'u': u_i, 'gt': gt_i,
#     })

# # --- Step 3: Optuna objective -- short partial fit per (trial, tuning image),
# # scored by PSNR - 10*HFEN, averaged over the tuning set.
# def huber_tv_objective(trial):
#     global LOSS_TYPE, HUBER_DELTA, TV_WEIGHT
#     LOSS_TYPE = "huber"
#     HUBER_DELTA = trial.suggest_float("huber_delta", DELTA_LOW, DELTA_HIGH, log=True)
#     TV_WEIGHT = trial.suggest_float("tv_weight", TV_LOW, TV_HIGH, log=True)

#     scores = []
#     for d in _tuning_data:
#         net_t = build_network({'output_depth': d['output_depth'], 'out_size': d['out_size']})
#         ni_t = get_z(d['u']['zf_complex_cropped'])
#         scaling_factor_t, ni_t = get_scale_factor(net_t, num_channels, in_size, d['u']['masked_kspace'], d['u']['mps'], ni=ni_t)
#         unders_measurement_t = Variable((d['u']['masked_kspace'] * scaling_factor_t)[None, :]).type(dtype)

#         _, net_t_fitted = fit(
#             net=net_t, img_noisy_var=unders_measurement_t, num_channels=num_channels,
#             net_input=ni_t, apply_f=forwardm, mask=d['u']['mask2d'], mask1d=d['u']['mask1d'],
#             scaling_factor=scaling_factor_t, num_iter=TUNING_PROBE_ITERS, LR=0.01, checkpoint_dir=None,
#         )
#         rec_t = normalize(reconstruct(net_t_fitted, ni_t, d['u']['mps']).astype(np.float64))
#         scores.append(psnr(d['gt'], rec_t) - 10 * hfen(d['gt'], rec_t))

#     return float(np.mean(scores))

# study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
# study.optimize(huber_tv_objective, n_trials=N_OPTUNA_TRIALS)

# HUBER_DELTA = study.best_params["huber_delta"]
# TV_WEIGHT = study.best_params["tv_weight"]
# LOSS_TYPE = "huber"
# print(f"Best trial: PSNR - 10*HFEN = {study.best_value:.4f}")
# print(f"  HUBER_DELTA = {HUBER_DELTA:.4g}")
# print(f"  TV_WEIGHT   = {TV_WEIGHT:.4g}")
# print("These globals are now set directly (Section 2's Config cell doesn't need to be "
#       "re-run for this), but call assert_run_tag_current() before fitting/saving anything "
#       "downstream, since RUN_TAG depends on them.")


## 7. Reference Image — Full Fit (Random Initialization)

This is the slow, from-scratch fit on `file1000033.h5`, using whichever `Z_SOURCE`/`ARCHITECTURE` you set above. Its converged weights become the **guided-init checkpoint** for Section 8, tagged as `{RUN_TAG}` so it never collides with a checkpoint from a different toggle configuration.

In [ ]:
assert_run_tag_current()  # catch a stale RUN_TAG before fitting/saving anything

ref = next(s for s in samples if s['filename'] == 'file1000033.h5')
slice_ksp = ref['slice_ksp']
slice_ksp_torchtensor = ref['slice_ksp_torchtensor']

output_depth = slice_ksp_torchtensor.numpy().shape[0] * 2
out_size = slice_ksp_torchtensor.numpy().shape[1:-1]
print(f"Number of coils: {slice_ksp_torchtensor.numpy().shape[0]}, "
      f"output_depth: {output_depth}, out_size: {out_size}")

ref_used_checkpoint = RUN_MODE == "load_existing" and os.path.exists(REF_CKPT_PATH)

if ref_used_checkpoint:
    print(f"RUN_MODE='load_existing': found {REF_CKPT_PATH} — loading instead of fitting.")
    checkpoint_ref = load_checkpoint_guarded(REF_CKPT_PATH)
    net = build_network({'output_depth': output_depth, 'out_size': out_size},
                         seed=checkpoint_ref.get('seed', SEED))
    net.load_state_dict(checkpoint_ref['model_state_dict'])
    ni = checkpoint_ref['net_input']
    mask, mask1d, mask2d = checkpoint_ref['mask'], checkpoint_ref['mask1d'], checkpoint_ref['mask2d']
    scaling_factor = checkpoint_ref['scaling_factor']

    masked_kspace, _ = apply_mask(slice_ksp_torchtensor, mask=mask)
    unders_measurement = Variable((masked_kspace * scaling_factor)[None, :]).type(dtype)
    mps = espirit_maps(masked_kspace)
    orig_tt = ifft2(masked_kspace)
    orig_np = orig_tt.cpu().numpy()
    orig_imgs_complex = orig_np[..., 0] + 1j * orig_np[..., 1]
    zf_img = np.abs(espirit_combine(orig_imgs_complex, mps))
    zf_img_cropped = crop_center(zf_img, 320, 320)
    gt_espirit = make_espirit_reference(slice_ksp, mps)
else:
    if RUN_MODE == "load_existing":
        print(f"RUN_MODE='load_existing' but no checkpoint found at {REF_CKPT_PATH} — falling back to a new fit.")
    net = build_network({'output_depth': output_depth, 'out_size': out_size})

    set_seed(SEED)  # reseed so the mask draw is identical regardless of architecture/z-source
    u = build_undersampled(slice_ksp_torchtensor, slice_ksp, net)
    mask, mask1d, mask2d = u['mask'], u['mask1d'], u['mask2d']
    zf_img_cropped = u['zf_img_cropped']
    mps = u['mps']
    gt_espirit = u['gt_espirit']

    ni = get_z(u['zf_complex_cropped'])
    scaling_factor, ni = get_scale_factor(net, num_channels, in_size, u['masked_kspace'], mps, ni=ni)
    unders_measurement = Variable((u['masked_kspace'] * scaling_factor)[None, :]).type(dtype)

show_difference(gt_espirit, zf_img_cropped, 'Undersampled (Zero-filled, 4x)')

### 7.1 Full fit (10,000 iterations)

Expect roughly an hour on a T4 GPU (faster on L4/A100). Checkpoints every 1000 iterations to Drive for resilience against disconnects.

In [ ]:
num_iters_slow = 10000

if not ref_used_checkpoint:
    mse_wrt_noisy, net = fit(
        net=net,
        img_noisy_var=unders_measurement,
        num_channels=num_channels,
        net_input=ni,
        apply_f=forwardm,
        mask=mask2d,
        mask1d=mask1d,
        scaling_factor=scaling_factor,
        num_iter=num_iters_slow,
        LR=0.01,
        checkpoint_dir=REF_CKPT_DIR,
        checkpoint_every=1000
    )
else:
    print("Skipping fit — reference network was loaded from a saved checkpoint.")

In [ ]:
if not ref_used_checkpoint:
    torch.save({
        'model_state_dict': net.state_dict(),
        'net_input': ni,
        'mask': mask, 'mask1d': mask1d, 'mask2d': mask2d,
        'scaling_factor': scaling_factor,
        'z_source': Z_SOURCE, 'z_model_id': (MRI_VAE_MODEL_ID if Z_SOURCE == 'mri_vae' else None), 'architecture': ARCHITECTURE, 'seed': SEED,
        'loss_type': LOSS_TYPE, 'huber_delta': HUBER_DELTA if LOSS_TYPE == "huber" else None,
        'tv_weight': TV_WEIGHT,
    }, REF_CKPT_PATH)
    print(f"Saved reference checkpoint ({RUN_TAG}) to {REF_CKPT_PATH}")
else:
    print(f"Skipping save — reference checkpoint already loaded from {REF_CKPT_PATH}")

In [ ]:
rec_cropped_ref = reconstruct(net, ni, mps)
show_full_comparison(gt_espirit, zf_img_cropped, rec_cropped_ref,
              f'ConvDecoder ({num_iters_slow} iter, random weights + {Z_LABEL}, {LOSS_LABEL})')

## 8. Comparing Initialization Strategies on a Second Image

All three runs below use the **same image** (`file1000041.h5`) and the **same mask/undersampled measurement**, so the only thing that differs between them is initialization and iteration budget. Loading the checkpoint from Section 7 includes a guard: if you switched `Z_SOURCE` since that checkpoint was built without re-running Section 7, this cell fails loudly instead of silently fitting the wrong combination.

- **Run A — random init, 10,000 iter**: full-convergence baseline, no head start
- **Run B — guided init, 10,000 iter**: same generous budget as A, but starting from the reference checkpoint — tests whether guided-init also raises the *ceiling*, not just the speed
- **Run C — guided init, 1,000 iter**: matches the paper's 10x-faster comparison — tests whether guided-init reaches comparable quality to Run A in a fraction of the time

In [ ]:
assert_run_tag_current()  # catch a stale RUN_TAG before fitting/saving anything

ref2 = next(s for s in samples if s['filename'] == 'file1000041.h5')
slice_ksp2 = ref2['slice_ksp']
slice_ksp_torchtensor2 = ref2['slice_ksp_torchtensor']

output_depth2 = slice_ksp_torchtensor2.numpy().shape[0] * 2
out_size2 = slice_ksp_torchtensor2.numpy().shape[1:-1]

checkpoint = load_checkpoint_guarded(REF_CKPT_PATH)  # guided-init source for Run B/C below

# Build the mask/measurement ONCE using a throwaway network just for scale estimation,
# then reuse the identical mask/measurement for all three runs below so they're directly comparable.
net_for_scale = build_network({'output_depth': output_depth2, 'out_size': out_size2})
set_seed(SEED)  # reseed so the mask draw is identical regardless of architecture/z-source
u2 = build_undersampled(slice_ksp_torchtensor2, slice_ksp2, net_for_scale)
_, mask1d_2, mask2d_2 = u2['mask'], u2['mask1d'], u2['mask2d']
zf_img_cropped2 = u2['zf_img_cropped']
mps2 = u2['mps']
gt_espirit2 = u2['gt_espirit']

if K_AVERAGING:
    print("K_AVERAGING is True — Run A/B/C below will be skipped; this cell's shared "
          "mask/measurement for the eval image is still needed by Section 12.")

show_difference(gt_espirit2, zf_img_cropped2, 'Undersampled (Zero-filled, 4x)')

### 8.1 Run A — random init, 10,000 iterations (full-convergence baseline)

In [ ]:
num_iters_A = 10000

if K_AVERAGING:
    print("K_AVERAGING is True — skipping Run A (only Section 12 runs). "
          "Set K_AVERAGING = False to run Run A/B/C.")
    rec_A = None
else:
    runA_used_checkpoint = RUN_MODE == "load_existing" and os.path.exists(RUNA_CKPT_PATH)

    if runA_used_checkpoint:
        print(f"RUN_MODE='load_existing': found {RUNA_CKPT_PATH} — loading instead of fitting.")
        ckpt_A = load_checkpoint_guarded(RUNA_CKPT_PATH)
        net_A_fitted = build_network({'output_depth': output_depth2, 'out_size': out_size2},
                                      seed=ckpt_A.get('seed', SEED))
        net_A_fitted.load_state_dict(ckpt_A['model_state_dict'])
        ni_A = ckpt_A['net_input']
    else:
        if RUN_MODE == "load_existing":
            print(f"RUN_MODE='load_existing' but no checkpoint found at {RUNA_CKPT_PATH} — falling back to a new fit.")
        net_A = build_network({'output_depth': output_depth2, 'out_size': out_size2})
        ni_A = get_z(u2['zf_complex_cropped'])
        scaling_factor_A, ni_A = get_scale_factor(net_A, num_channels, in_size, u2['masked_kspace'], mps2, ni=ni_A)
        unders_measurement_A = Variable((u2['masked_kspace'] * scaling_factor_A)[None, :]).type(dtype)

        _, net_A_fitted = fit(
            net=net_A,
            img_noisy_var=unders_measurement_A,
            num_channels=num_channels,
            net_input=ni_A,
            apply_f=forwardm,
            mask=mask2d_2,
            mask1d=mask1d_2,
            scaling_factor=scaling_factor_A,
            num_iter=num_iters_A,
            LR=0.01,
            checkpoint_dir=RUNA_CKPT_DIR,
            checkpoint_every=1000
        )

        torch.save({
            'model_state_dict': net_A_fitted.state_dict(),
            'net_input': ni_A, 'mask1d': mask1d_2, 'mask2d': mask2d_2,
            'scaling_factor': scaling_factor_A,
            'z_source': Z_SOURCE, 'z_model_id': (MRI_VAE_MODEL_ID if Z_SOURCE == 'mri_vae' else None), 'architecture': ARCHITECTURE, 'seed': SEED,
            'loss_type': LOSS_TYPE, 'huber_delta': HUBER_DELTA if LOSS_TYPE == "huber" else None,
            'tv_weight': TV_WEIGHT,
        }, RUNA_CKPT_PATH)

    rec_A = reconstruct(net_A_fitted, ni_A, mps2)
    show_full_comparison(gt_espirit2, zf_img_cropped2, rec_A, f'Run A: random init + {Z_LABEL}, {LOSS_LABEL}, {num_iters_A} iter')

### 8.2 Run B — guided init, 10,000 iterations (same budget as A — tests the ceiling)

In [ ]:
num_iters_B = 10000

if K_AVERAGING:
    print("K_AVERAGING is True — skipping Run B (only Section 12 runs). "
          "Set K_AVERAGING = False to run Run A/B/C.")
    rec_B = None
else:
    runB_used_checkpoint = RUN_MODE == "load_existing" and os.path.exists(RUNB_CKPT_PATH)

    if runB_used_checkpoint:
        print(f"RUN_MODE='load_existing': found {RUNB_CKPT_PATH} — loading instead of fitting.")
        ckpt_B = load_checkpoint_guarded(RUNB_CKPT_PATH)
        net_B_fitted = build_network({'output_depth': output_depth2, 'out_size': out_size2},
                                      seed=ckpt_B.get('seed', SEED))
        net_B_fitted.load_state_dict(ckpt_B['model_state_dict'])
        ni_B = ckpt_B['net_input']
    else:
        if RUN_MODE == "load_existing":
            print(f"RUN_MODE='load_existing' but no checkpoint found at {RUNB_CKPT_PATH} — falling back to a new fit.")
        net_B = build_network({'output_depth': output_depth2, 'out_size': out_size2})
        net_B.load_state_dict(checkpoint['model_state_dict'])  # guided init from the reference fit
        ni_B = get_z(u2['zf_complex_cropped'])
        scaling_factor_B, ni_B = get_scale_factor(net_B, num_channels, in_size, u2['masked_kspace'], mps2, ni=ni_B)
        unders_measurement_B = Variable((u2['masked_kspace'] * scaling_factor_B)[None, :]).type(dtype)

        _, net_B_fitted = fit(
            net=net_B,
            img_noisy_var=unders_measurement_B,
            num_channels=num_channels,
            net_input=ni_B,
            apply_f=forwardm,
            mask=mask2d_2,
            mask1d=mask1d_2,
            scaling_factor=scaling_factor_B,
            num_iter=num_iters_B,
            LR=0.01,
            checkpoint_dir=RUNB_CKPT_DIR,
            checkpoint_every=1000
        )

        torch.save({
            'model_state_dict': net_B_fitted.state_dict(),
            'net_input': ni_B, 'mask1d': mask1d_2, 'mask2d': mask2d_2,
            'scaling_factor': scaling_factor_B,
            'z_source': Z_SOURCE, 'z_model_id': (MRI_VAE_MODEL_ID if Z_SOURCE == 'mri_vae' else None), 'architecture': ARCHITECTURE, 'seed': SEED,
            'loss_type': LOSS_TYPE, 'huber_delta': HUBER_DELTA if LOSS_TYPE == "huber" else None,
            'tv_weight': TV_WEIGHT,
        }, RUNB_CKPT_PATH)

    rec_B = reconstruct(net_B_fitted, ni_B, mps2)
    show_full_comparison(gt_espirit2, zf_img_cropped2, rec_B, f'Run B: guided init + {Z_LABEL}, {LOSS_LABEL}, {num_iters_B} iter')

### 8.3 Run C — guided init, 1,000 iterations (paper's "10x faster" comparison)

In [ ]:
num_iters_C = num_iters_A // 10

if K_AVERAGING:
    print("K_AVERAGING is True — skipping Run C (only Section 12 runs). "
          "Set K_AVERAGING = False to run Run A/B/C.")
    rec_C = None
else:
    runC_used_checkpoint = RUN_MODE == "load_existing" and os.path.exists(RUNC_CKPT_PATH)

    if runC_used_checkpoint:
        print(f"RUN_MODE='load_existing': found {RUNC_CKPT_PATH} — loading instead of fitting.")
        ckpt_C = load_checkpoint_guarded(RUNC_CKPT_PATH)
        net_C_fitted = build_network({'output_depth': output_depth2, 'out_size': out_size2},
                                      seed=ckpt_C.get('seed', SEED))
        net_C_fitted.load_state_dict(ckpt_C['model_state_dict'])
        ni_C = ckpt_C['net_input']
    else:
        if RUN_MODE == "load_existing":
            print(f"RUN_MODE='load_existing' but no checkpoint found at {RUNC_CKPT_PATH} — falling back to a new fit.")
        net_C = build_network({'output_depth': output_depth2, 'out_size': out_size2})
        net_C.load_state_dict(checkpoint['model_state_dict'])  # guided init from the reference fit
        ni_C = get_z(u2['zf_complex_cropped'])
        scaling_factor_C, ni_C = get_scale_factor(net_C, num_channels, in_size, u2['masked_kspace'], mps2, ni=ni_C)
        unders_measurement_C = Variable((u2['masked_kspace'] * scaling_factor_C)[None, :]).type(dtype)

        _, net_C_fitted = fit(
            net=net_C,
            img_noisy_var=unders_measurement_C,
            num_channels=num_channels,
            net_input=ni_C,
            apply_f=forwardm,
            mask=mask2d_2,
            mask1d=mask1d_2,
            scaling_factor=scaling_factor_C,
            num_iter=num_iters_C,
            LR=0.01,
            checkpoint_dir=RUNC_CKPT_DIR,
            checkpoint_every=200
        )

        torch.save({
            'model_state_dict': net_C_fitted.state_dict(),
            'net_input': ni_C, 'mask1d': mask1d_2, 'mask2d': mask2d_2,
            'scaling_factor': scaling_factor_C,
            'z_source': Z_SOURCE, 'z_model_id': (MRI_VAE_MODEL_ID if Z_SOURCE == 'mri_vae' else None), 'architecture': ARCHITECTURE, 'seed': SEED,
            'loss_type': LOSS_TYPE, 'huber_delta': HUBER_DELTA if LOSS_TYPE == "huber" else None,
            'tv_weight': TV_WEIGHT,
        }, RUNC_CKPT_PATH)

    rec_C = reconstruct(net_C_fitted, ni_C, mps2)
    show_full_comparison(gt_espirit2, zf_img_cropped2, rec_C, f'Run C: guided init + {Z_LABEL}, {LOSS_LABEL}, {num_iters_C} iter')

## 9. Results Table (this run)

Quantitative comparison for **this notebook run** (i.e. this `Z_SOURCE`/`ARCHITECTURE` combination), saved to `RESULTS_DIR` tagged by `RUN_TAG`. All images are normalized to `[0, 1]` (by their own max) before computing metrics.

**Higher is better:** PSNR, SSIM, MS-SSIM, VIF
**Lower is better:** NMSE, HFEN

This cell does NOT compute any Δ against another run — that happens automatically in Section 10 once more than one `Z_SOURCE`/`ARCHITECTURE` combination has been run and saved.

In [ ]:
import pandas as pd

results = []

# Tag every row with the loss config that produced it, so later comparisons can tell an MSE run
# apart from a Huber(+TV) run instead of silently averaging or dropping one of them.
loss_meta_fitted = dict(
    Loss_Type=LOSS_TYPE,
    Huber_Delta=HUBER_DELTA if LOSS_TYPE == "huber" else 'n/a',
    TV_Weight=TV_WEIGHT,
)
loss_meta_baseline = dict(Loss_Type='n/a', Huber_Delta='n/a', TV_Weight='n/a')

# --- Image 1 (file1000033) — reference image ---
gt1  = normalize(gt_espirit.astype(np.float64))
zf1  = normalize(zf_img_cropped.astype(np.float64))
rec1 = normalize(rec_cropped_ref.astype(np.float64))

m = compute_all_metrics(gt1, zf1)
m.update(Architecture=ARCHITECTURE, Image=ref['filename'], Method='Zero-filled', Z_Source='n/a',
         **loss_meta_baseline)
results.append(m)

m = compute_all_metrics(gt1, rec1)
m.update(Architecture=ARCHITECTURE, Image=ref['filename'],
         Method=f'ConvDecoder (random weights, {num_iters_slow} iter)', Z_Source=Z_SOURCE,
         **loss_meta_fitted)
results.append(m)

# --- Image 2 (file1000041) — zero-filled baseline is always cheap to compute (no fitting),
# and gt2/zf2 are reused by Section 12's ensemble-averaging evaluation, so these stay
# unconditional even when K_AVERAGING skips Run A/B/C below.
gt2  = normalize(gt_espirit2.astype(np.float64))
zf2  = normalize(zf_img_cropped2.astype(np.float64))

m = compute_all_metrics(gt2, zf2)
m.update(Architecture=ARCHITECTURE, Image=ref2['filename'], Method='Zero-filled', Z_Source='n/a',
         **loss_meta_baseline)
results.append(m)

if not K_AVERAGING:
    recA = normalize(rec_A.astype(np.float64))
    recB = normalize(rec_B.astype(np.float64))
    recC = normalize(rec_C.astype(np.float64))

    m = compute_all_metrics(gt2, recA)
    m.update(Architecture=ARCHITECTURE, Image=ref2['filename'],
             Method=f'Run A: random init, {num_iters_A} iter', Z_Source=Z_SOURCE,
             **loss_meta_fitted)
    results.append(m)

    m = compute_all_metrics(gt2, recB)
    m.update(Architecture=ARCHITECTURE, Image=ref2['filename'],
             Method=f'Run B: guided init, {num_iters_B} iter', Z_Source=Z_SOURCE,
             **loss_meta_fitted)
    results.append(m)

    m = compute_all_metrics(gt2, recC)
    m.update(Architecture=ARCHITECTURE, Image=ref2['filename'],
             Method=f'Run C: guided init, {num_iters_C} iter', Z_Source=Z_SOURCE,
             **loss_meta_fitted)
    results.append(m)
else:
    print("K_AVERAGING is True — Run A/B/C rows skipped in this run's results table "
          "(see Section 12 for the ConvDecoder-A ensemble result instead).")

metric_cols = ['PSNR', 'SSIM', 'MS-SSIM', 'VIF', 'NMSE', 'HFEN']
meta_cols = ['Architecture', 'Image', 'Method', 'Z_Source', 'Loss_Type', 'Huber_Delta', 'TV_Weight']
results_df = pd.DataFrame(results)[meta_cols + metric_cols]

results_path = f"{RESULTS_DIR}/{RUN_TAG}.csv"
results_df.to_csv(results_path, index=False)
print(f"Saved this run's results to {results_path}")

results_df.set_index(['Image', 'Method', 'Z_Source', 'Loss_Type']).round(4)

## 10. Cross-Run Comparison (Δ)

Auto-loads every saved results CSV for the **current `ARCHITECTURE`** and computes Δ between `Z_SOURCE` variants — no hardcoded baseline numbers. Run this notebook once with `Z_SOURCE = "uniform"` and once with `Z_SOURCE = "vae"` (same `ARCHITECTURE`, same images), then run this cell.

Positive Δ = VAE input better than uniform input (sign auto-flipped for NMSE/HFEN, since lower is better for those).

In [ ]:
COMPARE_Z_SOURCE = "uniform"       # hold z-source fixed so the comparison isolates architecture only
BASELINE_ARCHITECTURE = "convdecoder"

# Glob every saved results file rather than pattern-matching RUN_TAG in the filename - RUN_TAG can
# now carry a loss-config suffix (e.g. "..._huber_tv0.000157.csv"), which broke the old filename-
# suffix-based glob. Filtering happens below using the actual columns in each CSV instead, which is
# robust to whatever RUN_TAG looks like.
result_files = glob.glob(f"{RESULTS_DIR}/*.csv")
print(f"Found {len(result_files)} saved result file(s) total:")
for f in result_files:
    print(" -", os.path.basename(f))

if not result_files:
    print("\nNo saved runs found yet — run Sections 7-9 at least once first.")
else:
    all_runs = pd.concat([pd.read_csv(f) for f in result_files], ignore_index=True)
    higher_better = {'PSNR', 'SSIM', 'MS-SSIM', 'VIF'}

    # Restrict to this Z_Source AND the current loss config, so an MSE run for one architecture
    # never gets compared against (or averaged with) a Huber+TV run for the other.
    fitted = all_runs[(all_runs['Z_Source'] == COMPARE_Z_SOURCE) & (all_runs['Loss_Type'] == LOSS_TYPE)]
    if LOSS_TYPE == "huber":
        fitted = fitted[np.isclose(fitted['Huber_Delta'].astype(float), HUBER_DELTA)]
    fitted = fitted[np.isclose(fitted['TV_Weight'].astype(float), TV_WEIGHT)]

    pivot = fitted.pivot_table(index=['Image', 'Method'], columns='Architecture', values=metric_cols)

    architectures_present = set(pivot.columns.get_level_values(1))
    if {BASELINE_ARCHITECTURE, ARCHITECTURE}.issubset(architectures_present):
        delta = pivot.xs(ARCHITECTURE, axis=1, level=1) - pivot.xs(BASELINE_ARCHITECTURE, axis=1, level=1)
        for c in metric_cols:
            if c not in higher_better:
                delta[c] = -delta[c]
        delta.columns = [f'Δ {c}' for c in delta.columns]

        print(f"Δ columns: positive = '{ARCHITECTURE}' better than '{BASELINE_ARCHITECTURE}' "
              f"(sign normalized per metric), both at Z_Source='{COMPARE_Z_SOURCE}', "
              f"loss config='{LOSS_LABEL}'")
        display(delta.round(4))
    else:
        missing = {BASELINE_ARCHITECTURE, ARCHITECTURE} - architectures_present
        print(f"\nMissing architecture run(s) at Z_Source='{COMPARE_Z_SOURCE}' under loss config "
              f"'{LOSS_LABEL}': {missing}. Set ARCHITECTURE to each missing value in the Config cell "
              f"(keep Z_SOURCE='{COMPARE_Z_SOURCE}' and the same loss config), re-run Sections 7-9, "
              f"then re-run this cell.")

## 11. Cross-Loss-Config Comparison (Δ)

The Section 10 cell above compares `Architecture` values (SE vs. plain) at a fixed loss config; it degenerates to a self-comparison if `ARCHITECTURE == BASELINE_ARCHITECTURE`. This cell compares the *loss config* instead — MSE vs. the current `LOSS_LABEL` (Huber, optionally +TV) — holding `ARCHITECTURE` and `Z_Source` fixed, which is the axis you actually want when checking whether Huber/TV helped or hurt relative to your original MSE baseline for the same architecture.

In [ ]:
COMPARE_Z_SOURCE = "uniform"          # hold z-source fixed
COMPARE_ARCHITECTURE = ARCHITECTURE   # hold architecture fixed — compare loss configs instead
BASELINE_LOSS_LABEL = "MSE"

result_files = glob.glob(f"{RESULTS_DIR}/*.csv")
print(f"Found {len(result_files)} saved result file(s) total:")
for f in result_files:
    print(" -", os.path.basename(f))

if not result_files:
    print("\nNo saved runs found yet — run Sections 7-9 at least once first.")
else:
    all_runs = pd.concat([pd.read_csv(f) for f in result_files], ignore_index=True)
    higher_better = {'PSNR', 'SSIM', 'MS-SSIM', 'VIF'}

    fitted = all_runs[(all_runs['Z_Source'] == COMPARE_Z_SOURCE) &
                       (all_runs['Architecture'] == COMPARE_ARCHITECTURE)].copy()

    def _loss_label(row):
        if row['Loss_Type'] == 'mse':
            label = "MSE"
        else:
            label = f"Huber(delta={float(row['Huber_Delta']):g})"
        tv = float(row['TV_Weight'])
        if tv > 0:
            label += f"+TV({tv:g})"
        return label

    fitted['Loss_Config'] = fitted.apply(_loss_label, axis=1)
    pivot = fitted.pivot_table(index=['Image', 'Method'], columns='Loss_Config', values=metric_cols)

    configs_present = set(pivot.columns.get_level_values(1))
    print(f"\nLoss configs found for architecture='{COMPARE_ARCHITECTURE}', "
          f"Z_Source='{COMPARE_Z_SOURCE}': {sorted(configs_present)}")

    if {BASELINE_LOSS_LABEL, LOSS_LABEL}.issubset(configs_present):
        delta = pivot.xs(LOSS_LABEL, axis=1, level=1) - pivot.xs(BASELINE_LOSS_LABEL, axis=1, level=1)
        for c in metric_cols:
            if c not in higher_better:
                delta[c] = -delta[c]
        delta.columns = [f'Δ {c}' for c in delta.columns]

        print(f"\nΔ columns: positive = '{LOSS_LABEL}' better than '{BASELINE_LOSS_LABEL}' "
              f"(sign normalized per metric), both at architecture='{COMPARE_ARCHITECTURE}', "
              f"Z_Source='{COMPARE_Z_SOURCE}'")
        display(delta.round(4))
    else:
        missing = {BASELINE_LOSS_LABEL, LOSS_LABEL} - configs_present
        print(f"\nMissing loss config(s): {missing}. Run Sections 7-9 under each missing config "
              f"(same ARCHITECTURE='{COMPARE_ARCHITECTURE}', Z_SOURCE='{COMPARE_Z_SOURCE}') first.")

## 12. Ensemble Averaging — ConvDecoder-A (k=5)

This reproduces the paper's ensemble-averaging result (their Fig. 5 / "ConvDecoder-A"): instead of a single accelerated (guided-init, 1,000-iteration) fit, we fit **K=5 independent accelerated reconstructions** of the same evaluation target (`file1000041.h5`) — each guided-init from a *different* reference checkpoint (i.e. a different source image's converged fit) — then average the K reconstructions pixel-wise.

This only applies to the accelerated regime (matches Run C's 1,000-iteration budget); it is not run for the slower random-init/full guided-init configurations.

**Why K different reference images, not just noise added to one reference?** If all K accelerated fits started from the *same* single guided-init checkpoint (Section 7's), they'd start from identical weights, and ConvDecoder's fitting here has no other source of randomness (no dropout, no minibatching) — so all K fits would converge to numerically identical reconstructions, and averaging them would do nothing. Using K distinct reference source images gives each ensemble member a genuinely different starting point, so their fit-specific errors are only partially correlated — which is exactly what averaging needs in order to cancel noise while reinforcing the shared true structure.

**Which K images are used as references:** chosen at random (not hand-picked) from every file in the data folder matching the evaluation target's acquisition type and coil count. This happens once per `RUN_TAG`/`K_VALUE` combination; the choice is then saved to disk and every later run (including after a runtime reset) reuses the same K filenames instead of drawing a new random set. Delete the saved selection file (its path is printed by this section's first cell) to force a fresh random draw.

**Compute cost:** each of the K reference checkpoints needs a full 10,000-iteration fit (same cost as Section 7), plus K accelerated 1,000-iteration fits. If `file1000033.h5` happens to be among the randomly chosen K, its checkpoint from Section 7 is reused directly as one ensemble member for free (only K-1 new reference fits needed); otherwise all K reference fits are new. At the established benchmarks (~31 min / ~65 min per full fit on L4/T4), expect roughly **K × ~31 min + K × ~3 min** in the worst case (no free reuse), or **(K-1) × ~31 min + K × ~3 min** if `file1000033.h5` is drawn, the first time this runs; with `RUN_MODE="load_existing"`, subsequent runs reload every cached checkpoint instead of refitting.

Set `K_AVERAGING = True` in Section 2 to run this section.

In [ ]:
if K_AVERAGING:
    assert_run_tag_current()

    EVAL_TARGET_FILENAME = 'file1000041.h5'  # must match the file used for Run A/B/C in Section 8

    def find_kavg_eligible_pool(exclude):
        '''Every file in `folder` matching EVAL_TARGET_FILENAME's acquisition type and coil
        count (so any candidate is guaranteed compatible with load_state_dict() as a guided-init
        source), excluding filenames in `exclude`. Shared by the initial random reference draw
        (below) and by the calibration-failure replacement logic in Section 12.1.'''
        with h5py.File(os.path.join(folder, EVAL_TARGET_FILENAME), 'r') as f:
            target_acquisition = f.attrs.get('acquisition')
            target_num_coils = f['kspace'].shape[1]
        pool = []
        for fname in all_files:
            if fname in exclude:
                continue
            try:
                with h5py.File(os.path.join(folder, fname), 'r') as f:
                    if (f.attrs.get('acquisition') != target_acquisition
                            or f['kspace'].shape[1] != target_num_coils):
                        continue
                    pool.append(fname)
            except OSError as e:
                print(f"  SKIPPED {fname}: {e}")
                continue
        return pool

    # The K reference source images are chosen ONCE (at random) and then persisted to disk, keyed
    # by RUN_TAG + K_VALUE, so every later run of this cell reuses the SAME K filenames instead of
    # redrawing a new random set. This matters because the per-file reference checkpoints cached in
    # Section 12.1 below are keyed by filename -- redrawing a different random set every run would
    # silently start fitting a different ensemble each time instead of reusing cached work.
    KAVG_REF_SELECTION_PATH = f"{CKPT_ROOT}/kavg_ref_selection_kavg{K_VALUE}_{RUN_TAG}.json"

    if os.path.exists(KAVG_REF_SELECTION_PATH):
        with open(KAVG_REF_SELECTION_PATH) as f:
            kavg_ref_filenames = json.load(f)
        print(f"Found existing reference selection at {KAVG_REF_SELECTION_PATH} -- reusing it "
              f"(delete this file if you want a fresh random draw instead).")
    else:
        print(f"No existing reference selection found at {KAVG_REF_SELECTION_PATH} -- drawing "
              f"{K_VALUE} random reference images now (this only happens once per RUN_TAG/K_VALUE "
              f"combination; the choice is then saved for every later run to reuse).")

        eligible_filenames = find_kavg_eligible_pool(exclude={EVAL_TARGET_FILENAME})

        assert len(eligible_filenames) >= K_VALUE, (
            f"Only found {len(eligible_filenames)} usable reference candidates in {folder}, "
            f"need {K_VALUE}. Add more files to the Drive folder or lower K_VALUE in Section 2."
        )

        # Random draw from the full eligible pool, using an independent RNG seeded from SEED (so
        # it's reproducible without perturbing the global `random` module state used elsewhere,
        # e.g. Section 6.5's tuning-image draw). file1000033.h5 is sorted first IF it happens to
        # be drawn, purely as a free efficiency win -- its reference checkpoint is already fit in
        # Section 7 regardless of K-averaging, so Section 12.1 can reuse it directly instead of
        # refitting. This does NOT bias which files get selected, only their fit order once chosen.
        rng = random.Random(SEED)
        kavg_ref_filenames = rng.sample(eligible_filenames, K_VALUE)
        kavg_ref_filenames.sort(key=lambda f: f != 'file1000033.h5')

        os.makedirs(os.path.dirname(KAVG_REF_SELECTION_PATH), exist_ok=True)
        with open(KAVG_REF_SELECTION_PATH, "w") as f:
            json.dump(kavg_ref_filenames, f, indent=1)
        print(f"Saved reference selection to {KAVG_REF_SELECTION_PATH} for reuse by future runs.")

    # Load any selected files not already in `samples` (a fresh runtime only has the 5 hand-picked
    # demo images loaded by Section 6; a reused selection from a prior run needs reloading here).
    loaded_filenames = {s['filename'] for s in samples}
    for fname in kavg_ref_filenames:
        if fname in loaded_filenames:
            continue
        with h5py.File(os.path.join(folder, fname), 'r') as f:
            slicenu = f["kspace"].shape[0] // 2
            slice_ksp = f['kspace'][slicenu]
            data = np.stack((slice_ksp.real, slice_ksp.imag), axis=-1)
            samples.append({
                'filename': fname,
                'slice_ksp': slice_ksp,
                'slice_ksp_torchtensor': torch.from_numpy(data),
            })

    print(f"\nUsing {K_VALUE} reference source images for ConvDecoder-A: {kavg_ref_filenames}")
else:
    print("K_AVERAGING is False — skipping Section 12 (set K_AVERAGING = True in Section 2 to run it).")


### 12.1 Fit (or reuse) K reference checkpoints

Builds (or loads from cache) the `K_VALUE` full 10,000-iteration guided-init reference fits for the randomly-chosen reference images selected above. If `file1000033.h5` happens to be among them, its checkpoint from Section 7 is reused directly as that member; the rest are fit fresh the first time, then cached permanently — this step ignores `RUN_MODE` on purpose (see the code comment) since these references have nothing to do with which image you're currently evaluating.

In [ ]:
if K_AVERAGING and KAVG_BUILD_REF_CHECKPOINTS:
    assert_run_tag_current()

    KAVG_TAG = f"kavg{K_VALUE}_{RUN_TAG}"
    KAVG_REF_CKPT_DIR = f"{CKPT_ROOT}/checkpoints_multicoil_{KAVG_TAG}_refs"
    KAVG_ACC_CKPT_DIR = f"{CKPT_ROOT}/checkpoints_multicoil_{KAVG_TAG}_acc"
    os.makedirs(KAVG_REF_CKPT_DIR, exist_ok=True)
    os.makedirs(KAVG_ACC_CKPT_DIR, exist_ok=True)

    # NOTE on RUN_MODE here: the K reference checkpoints are a shared, durable cache — each one
    # is expensive (10,000 iterations) but has nothing to do with *which* image you're currently
    # reconstructing. RUN_MODE is about the accelerated fit of the CURRENT eval target (Section
    # 12.2 below): "new_fit" should force a fresh accelerated reconstruction for a new image, but
    # it should NOT force these reference fits to redo work that's already sitting on disk. So,
    # unlike every other fit in this notebook, reference-checkpoint reuse here is existence-only
    # and deliberately ignores RUN_MODE — once a reference image has been fit, it's fit for good.

    kavg_ref_checkpoints = []  # one entry per ensemble member: {'filename', 'model_state_dict'}

    for idx, fname in enumerate(kavg_ref_filenames):
        if fname == 'file1000033.h5' and os.path.exists(REF_CKPT_PATH):
            # Reuse Section 7's reference fit directly instead of re-fitting it.
            print(f"[{idx}] {fname}: reusing Section 7 reference checkpoint ({REF_CKPT_PATH})")
            ckpt_i = load_checkpoint_guarded(REF_CKPT_PATH)
            kavg_ref_checkpoints.append({'filename': fname, 'model_state_dict': ckpt_i['model_state_dict']})
            continue

        stable_member_ckpt_path = f"{KAVG_REF_CKPT_DIR}/ref_{stable_stem(fname)}.pt"
        legacy_member_ckpt_path = f"{KAVG_REF_CKPT_DIR}/ref_{idx}_{fname}.pt"
        member_ckpt_path = first_existing_path(stable_member_ckpt_path, legacy_member_ckpt_path)

        if member_ckpt_path is not None:
            print(f"[{idx}] {fname}: reference checkpoint already exists — loading {member_ckpt_path}")
            ckpt_i = load_checkpoint_guarded(member_ckpt_path)
            kavg_ref_checkpoints.append({'filename': fname, 'model_state_dict': ckpt_i['model_state_dict']})
            continue

        print(f"[{idx}] {fname}: no cached reference checkpoint found — fitting now (10,000 iter).")

        already_appended = False
        u_i = None

        # If this image's ESPIRiT calibration fails (even after espirit_maps()'s own internal
        # retry with a larger calib_width), swap in a fresh replacement reference image instead
        # of letting one bad scan break the whole K-ensemble. The substitution is written back to
        # kavg_ref_filenames AND the persisted selection file, so it's permanent -- future runs,
        # and the batch-eval sections below (which also read kavg_ref_filenames), see the swap too.
        while u_i is None and not already_appended:
            sample_i = next(s for s in samples if s['filename'] == fname)
            ksp_i = sample_i['slice_ksp']
            ksp_tt_i = sample_i['slice_ksp_torchtensor']
            output_depth_i = ksp_tt_i.numpy().shape[0] * 2
            out_size_i = ksp_tt_i.numpy().shape[1:-1]

            net_i = build_network({'output_depth': output_depth_i, 'out_size': out_size_i})
            set_seed(SEED)
            try:
                u_i = build_undersampled(ksp_tt_i, ksp_i, net_i)
            except EspiritCalibrationError as e:
                print(f"    {fname}: ESPIRiT calibration failed ({e})")
                replacement_pool = find_kavg_eligible_pool(
                    exclude=set(kavg_ref_filenames) | {EVAL_TARGET_FILENAME, fname}
                )
                assert replacement_pool, (
                    f"No eligible replacement reference images left in {folder} after excluding "
                    f"the current K reference images, the eval target, and {fname} itself."
                )
                fname = replacement_pool[0]
                print(f"    Replacing with {fname} in the reference set.")
                kavg_ref_filenames[idx] = fname
                with open(KAVG_REF_SELECTION_PATH, "w") as jf:
                    json.dump(kavg_ref_filenames, jf, indent=1)
                print(f"    Saved updated reference selection to {KAVG_REF_SELECTION_PATH}.")

                if fname not in {s['filename'] for s in samples}:
                    with h5py.File(os.path.join(folder, fname), 'r') as f:
                        slicenu_new = f["kspace"].shape[0] // 2
                        slice_ksp_new = f['kspace'][slicenu_new]
                        data_new = np.stack((slice_ksp_new.real, slice_ksp_new.imag), axis=-1)
                        samples.append({
                            'filename': fname,
                            'slice_ksp': slice_ksp_new,
                            'slice_ksp_torchtensor': torch.from_numpy(data_new),
                        })

                stable_member_ckpt_path = f"{KAVG_REF_CKPT_DIR}/ref_{stable_stem(fname)}.pt"
                legacy_member_ckpt_path = f"{KAVG_REF_CKPT_DIR}/ref_{idx}_{fname}.pt"
                member_ckpt_path = first_existing_path(stable_member_ckpt_path, legacy_member_ckpt_path)
                if member_ckpt_path is not None:
                    print(f"    {fname}: reference checkpoint already exists — loading {member_ckpt_path}")
                    ckpt_i = load_checkpoint_guarded(member_ckpt_path)
                    kavg_ref_checkpoints.append({'filename': fname, 'model_state_dict': ckpt_i['model_state_dict']})
                    already_appended = True

        if already_appended:
            continue

        member_ckpt_path = stable_member_ckpt_path
        ni_i = get_z(u_i['zf_complex_cropped'])
        scaling_factor_i, ni_i = get_scale_factor(net_i, num_channels, in_size, u_i['masked_kspace'], u_i['mps'], ni=ni_i)
        unders_measurement_i = Variable((u_i['masked_kspace'] * scaling_factor_i)[None, :]).type(dtype)

        _, net_i_fitted = fit(
            net=net_i,
            img_noisy_var=unders_measurement_i,
            num_channels=num_channels,
            net_input=ni_i,
            apply_f=forwardm,
            mask=u_i['mask2d'],
            mask1d=u_i['mask1d'],
            scaling_factor=scaling_factor_i,
            num_iter=10000,
            LR=0.01,
            checkpoint_dir=f"{KAVG_REF_CKPT_DIR}/progress_{idx}",
            checkpoint_every=1000,
        )

        torch.save({
            'model_state_dict': net_i_fitted.state_dict(),
            'net_input': ni_i, 'mask': u_i['mask'], 'mask1d': u_i['mask1d'], 'mask2d': u_i['mask2d'],
            'scaling_factor': scaling_factor_i,
            **checkpoint_metadata(source_filename=fname, num_iterations=10000, checkpoint_role='k_reference'),
        }, member_ckpt_path)
        print(f"    saved {member_ckpt_path}")
        kavg_ref_checkpoints.append({'filename': fname, 'model_state_dict': net_i_fitted.state_dict()})

    print(f"\n{len(kavg_ref_checkpoints)} reference checkpoints ready: "
          f"{[c['filename'] for c in kavg_ref_checkpoints]}")
elif K_AVERAGING:
    print("KAVG_BUILD_REF_CHECKPOINTS is False -- skipping (no K-reference fitting this run).")

### 12.2 Accelerated (1,000-iteration) guided fit from each reference

For the single evaluation image `EVAL_TARGET_FILENAME`, fits one accelerated (1,000-iteration) guided reconstruction per reference checkpoint from 12.1 above — these `K_VALUE` reconstructions are what gets averaged in 12.3 below.

In [ ]:
if K_AVERAGING and KAVG_BUILD_REF_CHECKPOINTS:
    assert_run_tag_current()

    kavg_recons = []  # one reconstructed image per ensemble member

    for idx, ckpt_i in enumerate(kavg_ref_checkpoints):
        fname = ckpt_i['filename']
        member_acc_ckpt_path = f"{KAVG_ACC_CKPT_DIR}/acc_{idx}_{fname}.pt"

        if RUN_MODE == "load_existing" and os.path.exists(member_acc_ckpt_path):
            print(f"[{idx}] guided from {fname}: RUN_MODE='load_existing' — loading {member_acc_ckpt_path}")
            acc_ckpt_i = load_checkpoint_guarded(member_acc_ckpt_path)
            net_acc_i = build_network({'output_depth': output_depth2, 'out_size': out_size2},
                                       seed=acc_ckpt_i.get('seed', SEED))
            net_acc_i.load_state_dict(acc_ckpt_i['model_state_dict'])
            ni_acc_i = acc_ckpt_i['net_input']
        else:
            if RUN_MODE == "load_existing":
                print(f"[{idx}] guided from {fname}: no cached accelerated fit — fitting now.")
            else:
                print(f"[{idx}] guided from {fname}: fitting accelerated (1,000 iter) reconstruction ...")
            net_acc_i = build_network({'output_depth': output_depth2, 'out_size': out_size2})
            net_acc_i.load_state_dict(ckpt_i['model_state_dict'])  # guided init from ensemble member idx
            ni_acc_i = get_z(u2['zf_complex_cropped'])
            scaling_factor_acc_i, ni_acc_i = get_scale_factor(net_acc_i, num_channels, in_size,
                                                                u2['masked_kspace'], mps2, ni=ni_acc_i)
            unders_measurement_acc_i = Variable((u2['masked_kspace'] * scaling_factor_acc_i)[None, :]).type(dtype)

            _, net_acc_i_fitted = fit(
                net=net_acc_i,
                img_noisy_var=unders_measurement_acc_i,
                num_channels=num_channels,
                net_input=ni_acc_i,
                apply_f=forwardm,
                mask=mask2d_2,
                mask1d=mask1d_2,
                scaling_factor=scaling_factor_acc_i,
                num_iter=num_iters_C,   # same 1,000-iteration budget as Run C
                LR=0.01,
                checkpoint_dir=None,
            )
            net_acc_i = net_acc_i_fitted

            torch.save({
                'model_state_dict': net_acc_i.state_dict(),
                'net_input': ni_acc_i, 'mask1d': mask1d_2, 'mask2d': mask2d_2,
                'scaling_factor': scaling_factor_acc_i,
                'z_source': Z_SOURCE, 'z_model_id': (MRI_VAE_MODEL_ID if Z_SOURCE == 'mri_vae' else None), 'architecture': ARCHITECTURE, 'seed': SEED,
                'loss_type': LOSS_TYPE, 'huber_delta': HUBER_DELTA if LOSS_TYPE == "huber" else None,
                'tv_weight': TV_WEIGHT, 'source_filename': fname,
            }, member_acc_ckpt_path)
            print(f"    saved {member_acc_ckpt_path}")

        rec_i = reconstruct(net_acc_i, ni_acc_i, mps2)
        kavg_recons.append(rec_i)

    print(f"\n{len(kavg_recons)} accelerated reconstructions ready for averaging.")
elif K_AVERAGING:
    print("KAVG_BUILD_REF_CHECKPOINTS is False -- skipping (no K-reference fitting this run).")

### 12.3 Average and evaluate

Pixel-wise averages the `K_VALUE` accelerated reconstructions from 12.2 into the single ConvDecoder-A result, scores it against ground truth, and saves it to its own results CSV.

In [ ]:
if K_AVERAGING and KAVG_BUILD_REF_CHECKPOINTS:
    kavg_stack = np.stack(kavg_recons, axis=0)
    rec_kavg = kavg_stack.mean(axis=0)  # pixel-wise average of the K accelerated reconstructions

    # Reproduce the paper's Fig. 5 style curve: PSNR of the running average as k grows from 1..K.
    running_psnr = []
    for k in range(1, len(kavg_recons) + 1):
        running_avg = kavg_stack[:k].mean(axis=0)
        running_psnr.append(psnr(gt2, normalize(running_avg.astype(np.float64))))

    plt.figure(figsize=(5, 4))
    plt.plot(range(1, len(kavg_recons) + 1), running_psnr, marker='o')
    plt.xlabel('#ConvDecoders averaged (k)')
    plt.ylabel('PSNR')
    plt.title(f'Averaging {len(kavg_recons)} accelerated ({num_iters_C}-iter) reconstructions')
    plt.xticks(range(1, len(kavg_recons) + 1))
    plt.grid(alpha=0.3)
    plt.show()

    show_full_comparison(gt_espirit2, zf_img_cropped2, rec_kavg,
                          f'ConvDecoder-A (k={len(kavg_recons)}, guided init, {num_iters_C} iter each), {LOSS_LABEL}')
elif K_AVERAGING:
    print("KAVG_BUILD_REF_CHECKPOINTS is False -- skipping (no K-reference fitting this run).")

In [ ]:
if K_AVERAGING and KAVG_BUILD_REF_CHECKPOINTS:
    rec_kavg_n = normalize(rec_kavg.astype(np.float64))

    m_kavg = compute_all_metrics(gt2, rec_kavg_n)
    m_kavg.update(Architecture=ARCHITECTURE, Image=ref2['filename'],
                  Method=f'ConvDecoder-A (k={len(kavg_recons)}, guided init, {num_iters_C} iter)',
                  Z_Source=Z_SOURCE, Loss_Type=LOSS_TYPE,
                  Huber_Delta=HUBER_DELTA if LOSS_TYPE == "huber" else 'n/a', TV_Weight=TV_WEIGHT)

    kavg_results_df = pd.DataFrame([m_kavg])[meta_cols + metric_cols]
    kavg_results_path = f"{RESULTS_DIR}/{KAVG_TAG}.csv"
    kavg_results_df.to_csv(kavg_results_path, index=False)
    print(f"Saved ensemble-averaging result to {kavg_results_path}\n")

    # Direct comparison against this ensemble's own k=1 member (a single accelerated fit,
    # guided from kavg_ref_filenames[0], same iteration budget, same eval image) rather than
    # Section 8's Run C — Run C is skipped whenever K_AVERAGING is True, so it may not exist.
    m_k1 = compute_all_metrics(gt2, normalize(kavg_recons[0].astype(np.float64)))
    delta_row = {
        metric: (m_kavg[metric] - m_k1[metric]) if metric not in ('NMSE', 'HFEN')
                else (m_k1[metric] - m_kavg[metric])   # sign-normalize: positive = k-averaging better
        for metric in metric_cols
    }
    print(f"ConvDecoder-A (k={len(kavg_recons)}) vs. its own k=1 member "
          f"(single accelerated fit guided from {kavg_ref_filenames[0]}):")
    for metric in metric_cols:
        sign = '+' if delta_row[metric] >= 0 else ''
        print(f"  Δ{metric}: {sign}{delta_row[metric]:.4f}")

    display(kavg_results_df)
elif K_AVERAGING:
    print("KAVG_BUILD_REF_CHECKPOINTS is False -- skipping (no K-reference fitting this run).")

### 12.4 Qualitative benefit of averaging: k=1 vs. k={K}

Direct visual comparison between a single accelerated (guided-init, 1,000-iteration) fit — the same ensemble's own $k{=}1$ member — and the full $k$-averaged ConvDecoder-A reconstruction, so the averaging benefit is visible, not just numeric.

In [ ]:
if K_AVERAGING and KAVG_BUILD_REF_CHECKPOINTS:
    rec_k1_n = normalize(kavg_recons[0].astype(np.float64))
    gt_n2 = normalize(gt_espirit2)

    diff_k1 = np.abs(gt_n2 - rec_k1_n)
    diff_kavg = np.abs(gt_n2 - rec_kavg_n)

    # np.flipud(...) only for display -- see note in show_difference() (Section 5) above.
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    axes[0, 0].imshow(np.flipud(gt_espirit2), cmap='gray')
    axes[0, 0].set_title('Ground Truth'); axes[0, 0].axis('off')

    axes[0, 1].imshow(np.flipud(kavg_recons[0]), cmap='gray')
    axes[0, 1].set_title(f'k=1 (single guided fit, {num_iters_C} iter)\nguided from {kavg_ref_filenames[0]}')
    axes[0, 1].axis('off')

    axes[0, 2].imshow(np.flipud(rec_kavg), cmap='gray')
    axes[0, 2].set_title(f'ConvDecoder-A (k={len(kavg_recons)} average)')
    axes[0, 2].axis('off')

    axes[1, 0].imshow(np.flipud(zf_img_cropped2), cmap='gray')
    axes[1, 0].set_title('Zero-filled (Undersampled)'); axes[1, 0].axis('off')

    axes[1, 1].imshow(np.flipud(diff_k1), cmap='gray', vmin=0, vmax=DIFF_VMAX)
    axes[1, 1].set_title(f'Difference: k=1 (PSNR={m_k1["PSNR"]:.2f})'); axes[1, 1].axis('off')

    axes[1, 2].imshow(np.flipud(diff_kavg), cmap='gray', vmin=0, vmax=DIFF_VMAX)
    axes[1, 2].set_title(f'Difference: k={len(kavg_recons)} avg (PSNR={m_kavg["PSNR"]:.2f})'); axes[1, 2].axis('off')

    plt.suptitle(f'ConvDecoder-A ensemble benefit \u2014 {ARCHITECTURE}, {LOSS_LABEL}, {Z_LABEL}', fontsize=14)
    plt.tight_layout()
    plt.show()

    psnr_k1, psnr_kavg = m_k1['PSNR'], m_kavg['PSNR']
    ssim_k1, ssim_kavg = m_k1['SSIM'], m_kavg['SSIM']
    print(f"PSNR:    k=1 = {psnr_k1:.4f}   ->   k={len(kavg_recons)} avg = {psnr_kavg:.4f}   (\u0394 = {psnr_kavg - psnr_k1:+.4f})")
    print(f"SSIM:    k=1 = {ssim_k1:.4f}   ->   k={len(kavg_recons)} avg = {ssim_kavg:.4f}   (\u0394 = {ssim_kavg - ssim_k1:+.4f})")

elif K_AVERAGING:
    print("KAVG_BUILD_REF_CHECKPOINTS is False -- skipping (no K-reference fitting this run).")

### 12.5 Batch evaluation across `KAVG_BATCH_SIZE` images (paper-style mean ± std)

Reuses the exact same `K_VALUE` reference checkpoints fit/loaded in 12.1 above — no reference refitting here, regardless of `RUN_MODE` (same permanent-cache behavior as the single-image evaluation). For each of `KAVG_BATCH_SIZE` new evaluation images (auto-discovered, excluding the reference source images themselves and the single image evaluated in 12.3/12.4 above — guiding a fit from its own reference checkpoint would trivially inflate that image's metrics), this fits `K_VALUE` accelerated (1,000-iteration) guided reconstructions and averages them, exactly like 12.2/12.3 but looped over images. Each accelerated fit is itself cached per (image, reference) pair, so a disconnect only costs the fits since the last checkpoint, not the whole batch.

**Cost:** on an A100 (~10 min per 10k-iteration fit ⇒ ~1 min per accelerated 1,000-iteration fit), `KAVG_BATCH_SIZE=20` × `K_VALUE=10` = 200 accelerated fits ≈ 3h20m, since the reference checkpoints are already cached and reused as-is.

Set `RUN_BATCH_EVAL = True` in Section 2 to run this section.

In [ ]:
if K_AVERAGING and RUN_BATCH_EVAL:
    assert_run_tag_current()
    # Only kavg_ref_filenames (the reference image NAMES, from Section 12's preamble cell) is
    # actually used below -- kavg_ref_checkpoints (the FITTED weights, Section 12.1) is not needed
    # just to select batch targets, so this doesn't force you through K reference fits for a new
    # config if all you want from this config is Section 13's Run A/B/C batch eval.
    assert 'kavg_ref_filenames' in dir(), (
        "kavg_ref_filenames not found -- run Section 12's preamble cell (kavg01code) first, which "
        "only selects reference image names and does no fitting."
    )

    # Prune any manually-flagged images (KAVG_BATCH_MANUAL_EXCLUDE, set in Section 2) out of an
    # existing results CSV -- lets you drop a qualitatively-bad image after the fact and have a
    # replacement auto-selected below, without hand-editing the CSV.
    _batch_results_path_check = f"{RESULTS_DIR}/kavg_batch_{RUN_TAG}.csv"
    if KAVG_BATCH_MANUAL_EXCLUDE and os.path.exists(_batch_results_path_check):
        _df = pd.read_csv(_batch_results_path_check)
        _before = len(_df)
        _df = _df[~_df['Image'].isin(KAVG_BATCH_MANUAL_EXCLUDE)]
        if len(_df) != _before:
            _df.to_csv(_batch_results_path_check, index=False)
            print(f"Pruned {_before - len(_df)} manually-excluded image(s) from "
                  f"{_batch_results_path_check}: {sorted(KAVG_BATCH_MANUAL_EXCLUDE)}")

    # Exclude the K reference source images themselves (guiding a fit from its own reference
    # checkpoint would trivially near-perfectly reconstruct that image and inflate its metrics),
    # the single image already evaluated in 12.3/12.4 above, and any manually-flagged images
    # (KAVG_BATCH_MANUAL_EXCLUDE), so the batch is disjoint from all three and a dropped image
    # is never re-selected as its own replacement.
    exclude_from_batch = set(kavg_ref_filenames) | {EVAL_TARGET_FILENAME} | set(KAVG_BATCH_MANUAL_EXCLUDE)

    with h5py.File(os.path.join(folder, kavg_ref_filenames[0]), 'r') as f:
        target_acquisition = f.attrs.get('acquisition')
        target_num_coils = f['kspace'].shape[1]

    batch_target_filenames = []
    for fname in all_files:
        if len(batch_target_filenames) >= KAVG_BATCH_SIZE:
            break
        if fname in exclude_from_batch:
            continue
        try:
            with h5py.File(os.path.join(folder, fname), 'r') as f:
                if (f.attrs.get('acquisition') != target_acquisition
                        or f['kspace'].shape[1] != target_num_coils):
                    continue
                # Touch k-space shape to catch files whose header passes but whose data object
                # is malformed, before a long batch run starts.
                _ = f['kspace'].shape
        except (OSError, KeyError) as e:
            print(f"  SKIPPED {fname} (unreadable/corrupted): {e}")
            continue
        # Preflight the exact middle slice and ESPIRiT setup now. A failed candidate is
        # replaced immediately by the next eligible file, so the resulting list still has N images.
        try:
            with h5py.File(os.path.join(folder, fname), 'r') as f:
                _slice = f['kspace'][f['kspace'].shape[0] // 2]
            _ri = np.stack((_slice.real, _slice.imag), axis=-1)
            _ksp = torch.from_numpy(_ri)
            _net = build_network({'output_depth': _ksp.shape[0] * 2,
                                  'out_size': tuple(_ksp.shape[1:-1])})
            set_seed(SEED)
            _ = build_undersampled(_ksp, _slice, _net)
        except (OSError, KeyError, EspiritCalibrationError, ValueError) as e:
            print(f"  SKIPPED {fname} during preflight ({type(e).__name__}): {e}")
            continue
        batch_target_filenames.append(fname)

    assert len(batch_target_filenames) >= KAVG_BATCH_SIZE, (
        f"Only found {len(batch_target_filenames)} eligible batch targets in {folder}, need "
        f"{KAVG_BATCH_SIZE}. Add more files to the Drive folder or lower KAVG_BATCH_SIZE in Section 2."
    )

    print(f"Batch evaluation targets ({len(batch_target_filenames)}): {batch_target_filenames}")
else:
    print("RUN_BATCH_EVAL is False (or K_AVERAGING is False) -- skipping batch evaluation. "
          "Set K_AVERAGING = True and RUN_BATCH_EVAL = True in Section 2 to run it.")


In [ ]:
if K_AVERAGING and RUN_BATCH_EVAL and KAVG_BUILD_REF_CHECKPOINTS:
    assert_run_tag_current()

    KAVG_BATCH_ACC_CKPT_DIR = f"{CKPT_ROOT}/checkpoints_multicoil_{KAVG_TAG}_batch_acc"
    os.makedirs(KAVG_BATCH_ACC_CKPT_DIR, exist_ok=True)

    # Keyed by RUN_TAG only (not by the current KAVG_BATCH_SIZE) so a small test-run's
    # completed images accumulate into later larger runs instead of starting over.
    batch_results_path = f"{RESULTS_DIR}/kavg_batch_k{K_VALUE}_{RUN_TAG}.csv"
    legacy_batch_results_path = f"{RESULTS_DIR}/kavg_batch_{RUN_TAG}.csv"
    # Import a legacy results file only when every fitted row explicitly reports this K.
    if not os.path.exists(batch_results_path) and os.path.exists(legacy_batch_results_path):
        _legacy_df = pd.read_csv(legacy_batch_results_path)
        _legacy_methods = _legacy_df.get('Method', pd.Series(dtype=str)).astype(str)
        if len(_legacy_df) and _legacy_methods.str.contains(fr"\(k={K_VALUE},", regex=True).all():
            _legacy_df.to_csv(batch_results_path, index=False)
            print(f"Imported compatible legacy results into {batch_results_path}")
        elif len(_legacy_df):
            print(f"Ignoring {legacy_batch_results_path}: it is not explicitly a k={K_VALUE} result set.")

    # Resume support: if a prior (possibly crashed/disconnected) run already wrote partial
    # results, reload them and skip any image already completed instead of redoing it.
    if os.path.exists(batch_results_path):
        batch_results_df = pd.read_csv(batch_results_path)
        batch_rows = batch_results_df.to_dict('records')
        already_done = set(batch_results_df['Image'])
        print(f"Resuming from {batch_results_path}: {len(already_done)} image(s) already completed.")
    else:
        batch_rows = []
        already_done = set()

    skipped = []

    for img_idx, target_fname in enumerate(batch_target_filenames):
        if target_fname in already_done:
            print(f"\n=== Batch image {img_idx+1}/{len(batch_target_filenames)}: {target_fname} "
                  f"-- already completed, skipping ===")
            continue

        print(f"\n=== Batch image {img_idx+1}/{len(batch_target_filenames)}: {target_fname} ===")

        try:
            with h5py.File(os.path.join(folder, target_fname), 'r') as f:
                slicenu_t = f["kspace"].shape[0] // 2
                slice_ksp_t = f['kspace'][slicenu_t]
                data_t = np.stack((slice_ksp_t.real, slice_ksp_t.imag), axis=-1)
        except (OSError, KeyError) as e:
            print(f"  SKIPPED {target_fname} (unreadable/corrupted file): {type(e).__name__}: {e}")
            skipped.append(target_fname)
            continue

        ksp_tt_t = torch.from_numpy(data_t)
        output_depth_t = ksp_tt_t.numpy().shape[0] * 2
        out_size_t = ksp_tt_t.numpy().shape[1:-1]

        # Same recipe as Section 8's u2 setup (build_undersampled), applied to this batch image.
        try:
            net_for_scale_t = build_network({'output_depth': output_depth_t, 'out_size': out_size_t})
            set_seed(SEED)
            u_t = build_undersampled(ksp_tt_t, slice_ksp_t, net_for_scale_t)
        except (EspiritCalibrationError, ReconstructionDivergedError) as e:
            print(f"  SKIPPED {target_fname} (ESPIRiT calibration failed): {e}")
            skipped.append(target_fname)
            continue
        gt_t = normalize(u_t['gt_espirit'].astype(np.float64))

        # The whole ensemble-member loop below is wrapped so that if ANY member's fit()
        # diverges (ReconstructionDivergedError), the WHOLE IMAGE is abandoned and skipped --
        # same granularity as an ESPIRiT calibration failure above -- rather than silently
        # averaging in a diverged member's garbage reconstruction, or crashing the batch cell.
        try:
            member_recons = []
            for idx, ckpt_i in enumerate(kavg_ref_checkpoints):
                ref_fname = ckpt_i['filename']
                stable_member_ckpt_path = (
                    f"{KAVG_BATCH_ACC_CKPT_DIR}/target_{stable_stem(target_fname)}"
                    f"__ref_{stable_stem(ref_fname)}.pt"
                )
                legacy_member_ckpt_path = (
                    f"{KAVG_BATCH_ACC_CKPT_DIR}/acc_img{img_idx}_{target_fname}_ref{idx}_{ref_fname}.pt"
                )
                member_ckpt_path = first_existing_path(stable_member_ckpt_path, legacy_member_ckpt_path)

                if RUN_MODE == "load_existing" and member_ckpt_path is not None:
                    print(f"    [{idx+1}/{len(kavg_ref_checkpoints)}] loading cached fit: {member_ckpt_path}")
                    acc_ckpt = load_checkpoint_guarded(
                        member_ckpt_path, target_filename=target_fname, source_filename=ref_fname,
                        num_iterations=num_iters_C, k_value=K_VALUE,
                    )
                    net_acc = build_network({'output_depth': output_depth_t, 'out_size': out_size_t},
                                             seed=acc_ckpt.get('seed', SEED))
                    net_acc.load_state_dict(acc_ckpt['model_state_dict'])
                    ni_acc = acc_ckpt['net_input']
                else:
                    member_ckpt_path = stable_member_ckpt_path
                    print(f"    [{idx+1}/{len(kavg_ref_checkpoints)}] fitting {num_iters_C} iterations")
                    net_acc = build_network({'output_depth': output_depth_t, 'out_size': out_size_t})
                    net_acc.load_state_dict(ckpt_i['model_state_dict'])  # guided init from ensemble member idx
                    ni_acc = get_z(u_t['zf_complex_cropped'])
                    scaling_factor_acc, ni_acc = get_scale_factor(net_acc, num_channels, in_size,
                                                                    u_t['masked_kspace'], u_t['mps'], ni=ni_acc)
                    unders_measurement_acc = Variable((u_t['masked_kspace'] * scaling_factor_acc)[None, :]).type(dtype)

                    _, net_acc_fitted = fit(
                        net=net_acc,
                        img_noisy_var=unders_measurement_acc,
                        num_channels=num_channels,
                        net_input=ni_acc,
                        apply_f=forwardm,
                        mask=u_t['mask2d'],
                        mask1d=u_t['mask1d'],
                        scaling_factor=scaling_factor_acc,
                        num_iter=num_iters_C,
                        LR=0.01,
                        checkpoint_dir=None,
                    )
                    net_acc = net_acc_fitted

                    torch.save({
                        'model_state_dict': net_acc.state_dict(),
                        'net_input': ni_acc, 'mask1d': u_t['mask1d'], 'mask2d': u_t['mask2d'],
                        'scaling_factor': scaling_factor_acc,
                        **checkpoint_metadata(source_filename=ref_fname, target_filename=target_fname,
                                              num_iterations=num_iters_C, k_value=K_VALUE,
                                              checkpoint_role='k_batch_member'),
                    }, member_ckpt_path)
                    print(f"    [{idx}] guided from {ref_fname}: fit + saved {member_ckpt_path}")

                member_recons.append(reconstruct(net_acc, ni_acc, u_t['mps']))
        except ReconstructionDivergedError as e:
            print(f"  SKIPPED {target_fname} (reconstruction diverged during member fit): {e}")
            skipped.append(target_fname)
            continue


        rec_avg_n = normalize(np.stack(member_recons, axis=0).mean(axis=0).astype(np.float64))
        m_t = compute_all_metrics(gt_t, rec_avg_n)
        m_t.update(Architecture=ARCHITECTURE, Image=target_fname,
                    Method=f'ConvDecoder-A (k={len(kavg_ref_checkpoints)}, guided init, {num_iters_C} iter)',
                    Z_Source=Z_SOURCE, Loss_Type=LOSS_TYPE,
                    Huber_Delta=HUBER_DELTA if LOSS_TYPE == "huber" else 'n/a', TV_Weight=TV_WEIGHT)
        batch_rows.append(m_t)
        print(f"  {target_fname}: PSNR={m_t['PSNR']:.4f}  SSIM={m_t['SSIM']:.4f}")

        # Save after EVERY image (not just at the end) so a crash, corrupted file, or
        # disconnect never loses already-completed work -- re-running this cell resumes
        # from here via `already_done` above.
        batch_results_df = pd.DataFrame(batch_rows)[meta_cols + metric_cols]
        batch_results_df.to_csv(batch_results_path, index=False)

    print(f"\n{len(batch_rows)}/{len(batch_target_filenames)} image(s) completed. "
          f"Saved to {batch_results_path}")
    if skipped:
        print(f"Skipped {len(skipped)} image(s) (unreadable file or failed ESPIRiT calibration): {skipped}")
        print("Add these filenames to KAVG_BATCH_MANUAL_EXCLUDE (Section 2) and re-run Section "
              "12.5.1 to auto-backfill replacements, then re-run this cell.")
    display(batch_results_df)

elif K_AVERAGING and RUN_BATCH_EVAL:
    print("KAVG_BUILD_REF_CHECKPOINTS is False -- skipping (no K-averaged batch fitting this run).")

In [ ]:
if K_AVERAGING and RUN_BATCH_EVAL and KAVG_BUILD_REF_CHECKPOINTS:
    batch_summary = batch_results_df[metric_cols].agg(['mean', 'std']).T
    batch_summary.columns = ['Mean', 'Std']
    batch_summary['N'] = len(batch_results_df)

    # Named by the actual accumulated image count (batch_results_df may include images from
    # earlier smaller-KAVG_BATCH_SIZE test runs folded in via the shared batch_results_path).
    summary_path = f"{RESULTS_DIR}/kavg_batch{len(batch_results_df)}_{RUN_TAG}_summary.csv"
    batch_summary.to_csv(summary_path)
    print(f"Saved summary (mean +/- std over {len(batch_results_df)} images) to {summary_path}\n")

    print(f"ConvDecoder-A (k={K_VALUE}), {ARCHITECTURE}, {LOSS_LABEL}, {Z_LABEL} -- "
          f"mean +/- std over {len(batch_results_df)} images:")
    for metric in metric_cols:
        mean_v = batch_summary.loc[metric, 'Mean']
        std_v = batch_summary.loc[metric, 'Std']
        print(f"  {metric}: {mean_v:.4f} +/- {std_v:.4f}")

    display(batch_summary.round(4))

elif K_AVERAGING and RUN_BATCH_EVAL:
    print("KAVG_BUILD_REF_CHECKPOINTS is False -- skipping (no K-averaged batch fitting this run).")

## 13. Run A/B/C batch evaluation across N images (mean $\pm$ std)

This is **not** k-averaging -- each image gets exactly one Run A, one Run B, and one Run C fit (the same single fits Section 8 does for one image), just repeated over many images so we get paper-style mean $\pm$ std per run instead of a single-image number. It reuses `batch_target_filenames` from Section 12.5.1 as-is (same images already vetted there for readability/corruption), so Run A/B/C's batch results land on the *exact same images* as the K-averaged ConvDecoder-A batch result -- a fair, paired comparison across all methods.

**Checkpoint reuse:** every (image, run) fit is cached individually under its own path and reused whenever `RUN_MODE="load_existing"`, exactly like Section 12's caching -- interrupting and re-running this section only fits whatever is still missing. The results CSV is also saved after every single fit, not just at the end, so a crash or corrupted file never loses already-completed work (same fix as Section 12.5.2).

**Cost:** up to 3 fits per image (Run A: 10,000 iter random init; Run B: 10,000 iter guided init; Run C: 1,000 iter guided init) -- roughly 21 min/image on A100, 13 min/image on G4, or ~4h20m/2h40m total for a 20-image batch on A100/G4 respectively, assuming none of it is cached yet. Use `ABC_BATCH_RUNS` in Section 2 to drop the expensive runs (e.g. `{"C"}` only) if you just want the cheap one.

Set `RUN_ABC_BATCH_EVAL = True` in Section 2 to run this section.

In [ ]:
if RUN_ABC_BATCH_EVAL:
    assert_run_tag_current()
    assert 'batch_target_filenames' in dir(), (
        "batch_target_filenames not found -- run Section 12.5.1 (K-averaging batch target "
        "selection) at least once first, even if K_AVERAGING is currently False. This section "
        "reuses that same image list so every method is compared on identical images."
    )

    ABC_BATCH_CKPT_DIR = f"{CKPT_ROOT}/checkpoints_multicoil_{RUN_TAG}_abcbatch"
    os.makedirs(ABC_BATCH_CKPT_DIR, exist_ok=True)

    abc_batch_results_path = f"{RESULTS_DIR}/abcbatch_{RUN_TAG}.csv"

    # Reuses the SAME KAVG_BATCH_MANUAL_EXCLUDE set from Section 2/12.5.1 -- if you dropped an
    # image there, it's dropped here too. Prune any of its rows already sitting in THIS section's
    # own results CSV, and skip it below even if it is still (temporarily) present in
    # batch_target_filenames (e.g. 12.5.1 hasn't been re-run yet to backfill a replacement).
    if KAVG_BATCH_MANUAL_EXCLUDE and os.path.exists(abc_batch_results_path):
        _df = pd.read_csv(abc_batch_results_path)
        _before = len(_df)
        _df = _df[~_df['Image'].isin(KAVG_BATCH_MANUAL_EXCLUDE)]
        if len(_df) != _before:
            _df.to_csv(abc_batch_results_path, index=False)
            print(f"Pruned {_before - len(_df)} row(s) for manually-excluded image(s) from "
                  f"{abc_batch_results_path}: {sorted(KAVG_BATCH_MANUAL_EXCLUDE)}")

    # Resume support: reload any (image, run) pairs already completed in a prior/interrupted run.
    if os.path.exists(abc_batch_results_path):
        abc_batch_results_df = pd.read_csv(abc_batch_results_path)
        abc_batch_rows = abc_batch_results_df.to_dict('records')
        abc_already_done = set(zip(abc_batch_results_df['Image'], abc_batch_results_df['Run']))
        print(f"Resuming from {abc_batch_results_path}: {len(abc_already_done)} (image, run) pair(s) already completed.")
    else:
        abc_batch_rows = []
        abc_already_done = set()

    abc_skipped = []
    RUN_ITER = {'A': num_iters_A, 'B': num_iters_B, 'C': num_iters_C}
    RUN_GUIDED = {'A': False, 'B': True, 'C': True}

    for img_idx, target_fname in enumerate(batch_target_filenames):
        if target_fname in KAVG_BATCH_MANUAL_EXCLUDE:
            print(f"\n=== Batch image {img_idx+1}/{len(batch_target_filenames)}: {target_fname} "
                  f"-- manually excluded, skipping ===")
            continue

        print(f"\n=== Batch image {img_idx+1}/{len(batch_target_filenames)}: {target_fname} ===")

        try:
            with h5py.File(os.path.join(folder, target_fname), 'r') as f:
                slicenu_t = f["kspace"].shape[0] // 2
                slice_ksp_t = f['kspace'][slicenu_t]
                data_t = np.stack((slice_ksp_t.real, slice_ksp_t.imag), axis=-1)
        except (OSError, KeyError) as e:
            print(f"  SKIPPED {target_fname} (unreadable/corrupted file): {type(e).__name__}: {e}")
            abc_skipped.append(target_fname)
            continue

        ksp_tt_t = torch.from_numpy(data_t)
        output_depth_t = ksp_tt_t.numpy().shape[0] * 2
        out_size_t = ksp_tt_t.numpy().shape[1:-1]

        try:
            net_for_scale_t = build_network({'output_depth': output_depth_t, 'out_size': out_size_t})
            set_seed(SEED)
            u_t = build_undersampled(ksp_tt_t, slice_ksp_t, net_for_scale_t)
        except (EspiritCalibrationError, ReconstructionDivergedError) as e:
            print(f"  SKIPPED {target_fname} (ESPIRiT calibration failed): {e}")
            abc_skipped.append(target_fname)
            continue
        gt_t = normalize(u_t['gt_espirit'].astype(np.float64))

        for run_name in ("A", "B", "C"):
            if run_name not in ABC_BATCH_RUNS:
                continue
            if (target_fname, run_name) in abc_already_done:
                print(f"  Run {run_name}: already completed, skipping")
                continue

            # Wrapped so that if this run's fit() diverges (ReconstructionDivergedError),
            # the WHOLE IMAGE is abandoned (break out of the A/B/C run loop, skip to the next
            # target_fname) rather than crashing the cell or silently keeping partial results
            # for this image's other runs.
            try:
                stable_member_ckpt_path = f"{ABC_BATCH_CKPT_DIR}/run{run_name}_target_{stable_stem(target_fname)}.pt"
                legacy_member_ckpt_path = f"{ABC_BATCH_CKPT_DIR}/run{run_name}_img{img_idx}_{target_fname}.pt"
                member_ckpt_path = first_existing_path(stable_member_ckpt_path, legacy_member_ckpt_path)
                num_iter = RUN_ITER[run_name]

                if RUN_MODE == "load_existing" and member_ckpt_path is not None:
                    print(f"  Run {run_name}: RUN_MODE='load_existing' — loading {member_ckpt_path}")
                    ckpt_r = load_checkpoint_guarded(member_ckpt_path, target_filename=target_fname,
                                                     run_name=run_name, num_iterations=num_iter)
                    net_r = build_network({'output_depth': output_depth_t, 'out_size': out_size_t},
                                           seed=ckpt_r.get('seed', SEED))
                    net_r.load_state_dict(ckpt_r['model_state_dict'])
                    ni_r = ckpt_r['net_input']
                else:
                    member_ckpt_path = stable_member_ckpt_path
                    if RUN_MODE == "load_existing":
                        print(f"  Run {run_name}: no cached fit — fitting now ({num_iter} iter).")
                    else:
                        print(f"  Run {run_name}: fitting ({num_iter} iter) ...")
                    net_r = build_network({'output_depth': output_depth_t, 'out_size': out_size_t})
                    if RUN_GUIDED[run_name]:
                        net_r.load_state_dict(checkpoint['model_state_dict'])  # guided init, same Section 7 reference
                    ni_r = get_z(u_t['zf_complex_cropped'])
                    scaling_factor_r, ni_r = get_scale_factor(net_r, num_channels, in_size,
                                                                u_t['masked_kspace'], u_t['mps'], ni=ni_r)
                    unders_measurement_r = Variable((u_t['masked_kspace'] * scaling_factor_r)[None, :]).type(dtype)

                    _, net_r_fitted = fit(
                        net=net_r,
                        img_noisy_var=unders_measurement_r,
                        num_channels=num_channels,
                        net_input=ni_r,
                        apply_f=forwardm,
                        mask=u_t['mask2d'],
                        mask1d=u_t['mask1d'],
                        scaling_factor=scaling_factor_r,
                        num_iter=num_iter,
                        LR=0.01,
                        checkpoint_dir=None,
                    )
                    net_r = net_r_fitted

                    torch.save({
                        'model_state_dict': net_r.state_dict(),
                        'net_input': ni_r, 'mask1d': u_t['mask1d'], 'mask2d': u_t['mask2d'],
                        'scaling_factor': scaling_factor_r,
                        **checkpoint_metadata(target_filename=target_fname, run_name=run_name,
                                              num_iterations=num_iter, checkpoint_role='abc_batch'),
                    }, member_ckpt_path)
                    print(f"    saved {member_ckpt_path}")

                rec_r = reconstruct(net_r, ni_r, u_t['mps'])
                rec_r_n = normalize(rec_r.astype(np.float64))
                m_r = compute_all_metrics(gt_t, rec_r_n)
                m_r.update(Architecture=ARCHITECTURE, Image=target_fname, Run=run_name,
                           Method=f'Run {run_name} ({"guided" if RUN_GUIDED[run_name] else "random"} init, {num_iter} iter)',
                           Z_Source=Z_SOURCE, Loss_Type=LOSS_TYPE,
                           Huber_Delta=HUBER_DELTA if LOSS_TYPE == "huber" else 'n/a', TV_Weight=TV_WEIGHT)
                abc_batch_rows.append(m_r)
                print(f"    Run {run_name}: PSNR={m_r['PSNR']:.4f}  SSIM={m_r['SSIM']:.4f}")

                # Save after EVERY (image, run) fit -- never lose completed work to a later crash or
                # corrupted file, same fix as Section 12.5.2.
                abc_batch_results_df = pd.DataFrame(abc_batch_rows)[['Run'] + meta_cols + metric_cols]
                abc_batch_results_df.to_csv(abc_batch_results_path, index=False)
            except ReconstructionDivergedError as e:
                print(f"  Run {run_name}: SKIPPED {target_fname} (reconstruction diverged): {e}")
                abc_skipped.append(target_fname)
                break


    print(f"\n{len(abc_batch_rows)} (image, run) result(s) total. Saved to {abc_batch_results_path}")
    if abc_skipped:
        print(f"Skipped {len(abc_skipped)} image(s) (unreadable file or failed ESPIRiT calibration): {abc_skipped}")
        print("Add these filenames to KAVG_BATCH_MANUAL_EXCLUDE (Section 2) and re-run Section "
              "12.5.1 to auto-backfill replacements, then re-run this cell.")
    display(abc_batch_results_df)
else:
    print("RUN_ABC_BATCH_EVAL is False -- skipping Section 13.")

In [ ]:
if RUN_ABC_BATCH_EVAL:
    print(f"Run A/B/C batch evaluation -- {ARCHITECTURE}, {LOSS_LABEL}, {Z_LABEL} -- "
          f"mean +/- std per run:\n")

    abc_summary_rows = []
    for run_name, num_iter, init_desc in (("A", num_iters_A, "random init"),
                                           ("B", num_iters_B, "guided init"),
                                           ("C", num_iters_C, "guided init")):
        sub = abc_batch_results_df[abc_batch_results_df['Run'] == run_name]
        if sub.empty:
            continue
        print(f"Run {run_name} ({init_desc}, {num_iter} iter), n={len(sub)}:")
        row = {'Run': run_name, 'N': len(sub)}
        for metric in metric_cols:
            mean_v, std_v = sub[metric].mean(), sub[metric].std()
            row[f'{metric}_mean'] = mean_v
            row[f'{metric}_std'] = std_v
            print(f"  {metric}: {mean_v:.4f} +/- {std_v:.4f}")
        abc_summary_rows.append(row)
        print()

    abc_summary_df = pd.DataFrame(abc_summary_rows)
    abc_summary_path = f"{RESULTS_DIR}/abcbatch_{RUN_TAG}_summary.csv"
    abc_summary_df.to_csv(abc_summary_path, index=False)
    print(f"Saved summary to {abc_summary_path}")
    display(abc_summary_df.round(4))

## 14. Delete selected fitted images safely
Preview first; this supports both legacy and stable checkpoint names.


In [ ]:
# ============================================================
# CHECKPOINT / RESULT CLEANUP (preview first)
# ============================================================
DELETE_FITTED_IMAGES = set()  # e.g. {"file1000031.h5", "file1000033.h5"}
CONFIRM_DELETE_FITTED = False

assert_run_tag_current()
delete_names = {os.path.basename(x) for x in DELETE_FITTED_IMAGES}
delete_stems = {stable_stem(x) for x in delete_names}
paths_to_delete = set()

# Cover stable target_<stem> names and the older img<position>_<filename> names.
checkpoint_dirs = [
    globals().get('KAVG_BATCH_ACC_CKPT_DIR'),
    globals().get('ABC_BATCH_CKPT_DIR'),
    globals().get('KAVG_ACC_CKPT_DIR'),
]
for directory in filter(None, checkpoint_dirs):
    if not os.path.isdir(directory):
        continue
    for p in glob.glob(os.path.join(directory, '*.pt')):
        base = os.path.basename(p)
        if any(f"target_{stem}" in base for stem in delete_stems) or any(name in base for name in delete_names):
            paths_to_delete.add(p)

result_paths = [
    globals().get('batch_results_path'),
    globals().get('legacy_batch_results_path'),
    globals().get('abc_batch_results_path'),
]
result_edits = []
for csv_path in filter(None, result_paths):
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        if 'Image' in df.columns:
            n = int(df['Image'].isin(delete_names).sum())
            if n:
                result_edits.append((csv_path, df.loc[~df['Image'].isin(delete_names)].copy(), n))

summary_paths = set()
for pattern in (
    f"{RESULTS_DIR}/kavg_batch*_{RUN_TAG}_summary.csv",
    f"{RESULTS_DIR}/abcbatch_{RUN_TAG}_summary.csv",
):
    summary_paths.update(glob.glob(pattern))

print(f"Selected images: {sorted(delete_names)}")
print(f"Checkpoint files matched ({len(paths_to_delete)}):")
for p in sorted(paths_to_delete): print('  ', p)
print("CSV rows matched:")
for p, _, n in result_edits: print(f"  {p}: {n} row(s)")
print(f"Derived summaries to invalidate ({len(summary_paths)}):")
for p in sorted(summary_paths): print('  ', p)

if not CONFIRM_DELETE_FITTED:
    print("PREVIEW ONLY. Set CONFIRM_DELETE_FITTED=True and rerun this cell to delete saved fits/results.")
else:
    assert delete_names, "Refusing an empty deletion request."
    for p in sorted(paths_to_delete): os.remove(p)
    for p, df, _ in result_edits: df.to_csv(p, index=False)
    for p in sorted(summary_paths): os.remove(p)
    print(f"Deleted {len(paths_to_delete)} checkpoint(s), updated {len(result_edits)} CSV(s), "
          f"and invalidated {len(summary_paths)} summary file(s). Original .h5 files were not touched.")


## 15. Save & Push to GitHub

Run manually whenever you want to snapshot a milestone -- this is not part of the automatic
pipeline and nothing above depends on it. Keeps the project's checkpoint/results split intact:

- **Stays on Drive** (`CKPT_ROOT` / `RESULTS_DIR`): every `.pt` checkpoint, the K-averaging
  reference-selection JSON files, the full per-image batch results. These are large and fully
  regenerable from code + a fixed `RUN_TAG`/seed, so they never go into git.
- **Goes to GitHub**: this notebook (with cell outputs stripped, so diffs stay readable), the
  small results CSVs from `RESULTS_DIR`, and a `results/manifest.csv` row recording which config
  produced them.

To find which commit produced a given manifest row later, use
`git log --follow -p -- results/manifest.csv` (or `git blame results/manifest.csv`) in the repo --
simpler than trying to embed a not-yet-created commit hash into its own commit.

Requires a `GITHUB_TOKEN` Colab secret (see Section 1) with push access to
`itaipasternak-cloud/ConvDeconv-MRI-project`. Safe to skip entirely if you're not ready to push yet.


In [ ]:
# ============================================================
# Save & Push to GitHub -- run manually, not part of the main pipeline
# ============================================================
import subprocess, shutil, glob, csv, datetime

REPO_RESULTS_DIR = os.path.join(REPO_DIR, "results")
os.makedirs(REPO_RESULTS_DIR, exist_ok=True)
MANIFEST_PATH = os.path.join(REPO_RESULTS_DIR, "manifest.csv")

# 1. Copy this notebook in from Drive and strip its outputs before committing, so diffs stay
#    readable. This only affects what gets committed -- your live Colab session is untouched.
#    Update NOTEBOOK_SRC below if you saved this .ipynb somewhere other than Colab's default
#    "My Drive/Colab Notebooks" folder.
NOTEBOOK_NAME = "MRI_ConvDeconv_espirit.ipynb"
NOTEBOOK_SRC = f"/content/drive/MyDrive/Colab Notebooks/{NOTEBOOK_NAME}"
notebook_dst = os.path.join(REPO_DIR, NOTEBOOK_NAME)
if os.path.exists(NOTEBOOK_SRC):
    subprocess.run(["pip", "install", "-q", "nbstripout"], check=True)
    shutil.copy(NOTEBOOK_SRC, notebook_dst)
    subprocess.run(["python", "-m", "nbstripout", notebook_dst], check=True)
    print(f"Copied + stripped outputs: {notebook_dst}")
else:
    print(f"WARNING: {NOTEBOOK_SRC!r} not found -- skipping notebook copy. Set NOTEBOOK_SRC above "
          f"to wherever you actually saved this .ipynb, then re-run this cell.")

# 2. Copy small results CSVs (NOT checkpoints -- those stay on Drive) into the repo.
csv_paths = glob.glob(f"{RESULTS_DIR}/*.csv")
for csv_path in csv_paths:
    shutil.copy(csv_path, REPO_RESULTS_DIR)
print(f"Copied {len(csv_paths)} results CSV(s) into {REPO_RESULTS_DIR}")

# 3. Append one manifest row describing this run's config. The commit that contains this exact
#    row is whatever commit this cell makes below -- discoverable later via `git log`/`git blame`
#    on results/manifest.csv rather than trying to record a not-yet-existing hash.
manifest_row = {
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    "run_tag": RUN_TAG if "RUN_TAG" in dir() else "n/a",
    "z_source": Z_SOURCE,
    "architecture": ARCHITECTURE,
    "loss_type": LOSS_TYPE,
    "huber_delta": HUBER_DELTA if LOSS_TYPE == "huber" else "n/a",
    "tv_weight": TV_WEIGHT,
    "k_value": K_VALUE if "K_VALUE" in dir() else "n/a",
    "results_csvs_copied": ";".join(os.path.basename(p) for p in csv_paths),
}
manifest_is_new = not os.path.exists(MANIFEST_PATH)
with open(MANIFEST_PATH, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(manifest_row.keys()))
    if manifest_is_new:
        writer.writeheader()
    writer.writerow(manifest_row)
print(f"Appended manifest row to {MANIFEST_PATH}")

# 4. Commit and push.
subprocess.run(["git", "-C", REPO_DIR, "add", "-A"], check=True)
commit_msg = f"Update results/notebook -- {manifest_row['timestamp']} (RUN_TAG={manifest_row['run_tag']})"
commit_result = subprocess.run(["git", "-C", REPO_DIR, "commit", "-m", commit_msg],
                                capture_output=True, text=True)
print(commit_result.stdout.strip() or commit_result.stderr.strip())

push_url = _github_push_url()
push_result = subprocess.run(["git", "-C", REPO_DIR, "push", push_url, "HEAD:main"],
                              capture_output=True, text=True)
if push_result.returncode != 0:
    print("PUSH FAILED:\n", push_result.stderr)
else:
    commit_hash = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                                  capture_output=True, text=True).stdout.strip()
    print(f"Pushed commit {commit_hash[:10]} to {REPO_SLUG}.")
